# ScriptSense: AI-writing detector, end-to-end ML pipeline

This notebook contains the **complete `ml/` codebase** of ScriptSense: data preparation, deduplication and
leakage-safe splitting, multi-task DeBERTa training, calibration, evaluation, document inference and the first
similarity-engine components.

### How to train (Google Colab)
1. **Runtime → Change runtime type → T4 GPU** (or better) → Save.
2. **Runtime → Run all.** Allow Google Drive access when asked.
3. That's it. The notebook downloads the data, prepares it, runs a short smoke test, then trains the full model.
   Trained models, the prepared dataset and the training checkpoint are saved to
   **Google Drive → `MyDrive/scriptsense_models/`** (about 4–5 GB in total).

**If Colab disconnects:** reconnect, set the T4 GPU again, and **Run all** again. Finished steps are skipped,
the prepared dataset is restored from Drive, and training **resumes from its last checkpoint** (saved every
1,000 steps). It does not start over.

**Truth constraints:** scores are probabilistic signals, not proof of misconduct, and are not comparable to Turnitin.
Never report a target number as a measured result, and never tune on test/challenge data.

## 1. Environment setup
Run this first, including after any restart or disconnect.

In [ ]:
import os, sys, pathlib, subprocess, shutil, json
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
SAVE_TO_DRIVE = True   # Colab: keep models, checkpoint and prepared dataset on Google Drive

if IN_COLAB:
    os.environ["SCRIPTSENSE_DATA_DIR"] = "/content/scriptsense_data"   # raw data: too big for free Drive
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["SCRIPTSENSE_ARTIFACT_DIR"] = "/content/drive/MyDrive/scriptsense_models"
    WORK_DIR = pathlib.Path("/content/scriptsense")
else:
    # Local: a separate workspace so the repository's own files are never overwritten.
    # Raw/processed data defaults to ~/scriptsense_data, so already-downloaded datasets are reused.
    WORK_DIR = pathlib.Path.home() / "scriptsense_notebook"

WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
PY = sys.executable
DATA_DIR = pathlib.Path(os.environ.get("SCRIPTSENSE_DATA_DIR", pathlib.Path.home() / "scriptsense_data"))
ART_DIR = pathlib.Path(os.environ.get("SCRIPTSENSE_ARTIFACT_DIR", DATA_DIR / "artifacts"))
BACKUP_DIR = ART_DIR / "prepared"            # prepared dataset + manifests (restored after a disconnect)
DATASET = DATA_DIR / "processed" / "dataset.parquet"
ART_DIR.mkdir(parents=True, exist_ok=True)


def sh(*args, cwd=None):
    """Run a command with the notebook's Python, stream its output, and STOP the notebook if it fails."""
    p = subprocess.Popen([PY, *args], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed (exit code {p.returncode}): {' '.join(map(str, args))}")


def run_finished(name):
    meta = ART_DIR / "runs" / name / "run_meta.json"
    return meta.exists() and bool(json.loads(meta.read_text()).get("finished_at"))


print("work dir  :", WORK_DIR)
print("data dir  :", DATA_DIR)
print("model dir :", ART_DIR)

In [ ]:
# Create the package directories and empty __init__.py files
for d in ['ml/scriptsense_ml/data', 'ml/scriptsense_ml/models', 'ml/scriptsense_ml/training',
          'ml/scriptsense_ml/inference', 'ml/scriptsense_ml/evaluation', 'ml/scriptsense_ml/similarity',
          'ml/scripts', 'ml/configs', 'ml/tests', 'data/manifests']:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
for f in ['scriptsense_ml/__init__.py', 'scriptsense_ml/data/__init__.py', 'scriptsense_ml/evaluation/__init__.py', 'scriptsense_ml/inference/__init__.py', 'scriptsense_ml/models/__init__.py', 'scriptsense_ml/similarity/__init__.py', 'scriptsense_ml/training/__init__.py']:
    pathlib.Path('ml', f).touch()
print('ok')

## 2. Code: Package: paths and data layer

### `ml/scriptsense_ml/paths.py`
Filesystem locations for ScriptSense ML data.

In [ ]:
%%writefile ml/scriptsense_ml/paths.py
"""Filesystem locations for ScriptSense ML data.

Raw and processed datasets are large and/or licensed, so they live outside the
repository (which sits in a cloud-synced folder). Override with the
SCRIPTSENSE_DATA_DIR environment variable. Manifests are small and tracked.
"""
from __future__ import annotations

import os
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parents[2]
ML_ROOT = REPO_ROOT / "ml"

DATA_DIR = Path(os.environ.get("SCRIPTSENSE_DATA_DIR", Path.home() / "scriptsense_data"))
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MANIFEST_DIR = Path(os.environ.get("SCRIPTSENSE_MANIFEST_DIR", REPO_ROOT / "data" / "manifests"))
ARTIFACT_DIR = Path(os.environ.get("SCRIPTSENSE_ARTIFACT_DIR", DATA_DIR / "artifacts"))


def ensure_dirs() -> None:
    for d in (RAW_DIR, PROCESSED_DIR, MANIFEST_DIR, ARTIFACT_DIR):
        d.mkdir(parents=True, exist_ok=True)


### `ml/scriptsense_ml/data/schema.py`
Canonical row schema and label taxonomy shared by data, training, inference and API.

In [ ]:
%%writefile ml/scriptsense_ml/data/schema.py
"""Canonical row schema and label taxonomy shared by data, training, inference and API."""
from __future__ import annotations

from enum import IntEnum

UNKNOWN = -1  # label not supported by the source's provenance; excluded from that head's loss


class Class4(IntEnum):
    HUMAN_ONLY = 0
    AI_GENERATED = 1
    AI_PARAPHRASED_OR_REFINED = 2
    MIXED_OR_AI_ASSISTED = 3


CLASS4_NAMES = [c.name for c in Class4]
BINARY_NAMES = ["HUMAN_ONLY", "AI_INVOLVED"]

SPLITS = ("train", "validation", "test", "challenge")

# Columns every processed row must carry (CLAUDE.md section 4, data governance).
COLUMNS = [
    "row_id",               # globally unique: f"{source_dataset}:{source_row_id}"
    "text",                 # normalize_for_model(raw_text)
    "raw_text",
    "source_dataset",
    "source_row_id",
    "original_document_id",  # lineage root if known (e.g. RAID source_id)
    "prompt_id",
    "generator",
    "domain",
    "attack_or_edit_type",
    "provenance",           # short human-readable explanation of why the label holds
    "source_label",         # the dataset's original label, verbatim
    "class_4way",           # Class4 value or UNKNOWN
    "label_binary",         # 0/1 or UNKNOWN
    "split",                # fixed by the source (official dev/test) or "" until assigned
    "split_fixed",          # True when the split comes from the dataset publisher
    "group_keys",           # "|"-joined lineage keys that must not cross splits
    "license_reference",
]


def binary_from_class4(c4: int) -> int:
    if c4 == UNKNOWN:
        return UNKNOWN
    return 0 if c4 == Class4.HUMAN_ONLY else 1


def validate_row(row: dict) -> None:
    missing = [c for c in COLUMNS if c not in row]
    if missing:
        raise ValueError(f"row {row.get('row_id')} missing columns {missing}")
    if not isinstance(row["text"], str) or not row["text"].strip():
        raise ValueError(f"row {row['row_id']} has empty text")
    c4, b = row["class_4way"], row["label_binary"]
    if c4 not in (UNKNOWN, *iter(Class4)):
        raise ValueError(f"row {row['row_id']} bad class_4way {c4}")
    if b not in (UNKNOWN, 0, 1):
        raise ValueError(f"row {row['row_id']} bad label_binary {b}")
    if c4 != UNKNOWN and b != UNKNOWN and binary_from_class4(c4) != b:
        raise ValueError(f"row {row['row_id']} binary {b} inconsistent with class {c4}")
    if c4 == UNKNOWN and b == UNKNOWN:
        raise ValueError(f"row {row['row_id']} carries no usable label")


### `ml/scriptsense_ml/data/normalization.py`
Text normalization.

In [ ]:
%%writefile ml/scriptsense_ml/data/normalization.py
"""Text normalization.

Two levels, deliberately different:

* ``normalize_for_model`` is conservative and is applied identically at training
  and inference time. It keeps casing, punctuation and paragraph breaks, and only
  removes invisible characters that carry no linguistic content (zero-width
  spaces, BOMs, soft hyphens). Removing them is intentional: RAID's
  ``zero_width_space`` attack would otherwise be a trivially exploitable signal.
* ``dedup_key`` is aggressive (NFKC, casefold, punctuation-insensitive) and is
  used only to detect duplicates. It is never shown to a model or user.

Raw text is always stored separately so offsets can be mapped back.
"""
from __future__ import annotations

import hashlib
import re
import unicodedata

_INVISIBLE = dict.fromkeys(map(ord, "​‌‍⁠﻿­"), None)
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NEWLINES = re.compile(r"\n{3,}")
_SPACE_AROUND_NL = re.compile(r" *\n *")
_NON_WORD = re.compile(r"[^\w\s]", re.UNICODE)
_WS = re.compile(r"\s+")


def normalize_for_model(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.translate(_INVISIBLE)
    text = _HSPACE.sub(" ", text)
    text = _SPACE_AROUND_NL.sub("\n", text)
    text = _MANY_NEWLINES.sub("\n\n", text)
    return text.strip()


def dedup_key(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).translate(_INVISIBLE).casefold()
    text = _NON_WORD.sub(" ", text)
    return _WS.sub(" ", text).strip()


def text_hash(text: str) -> str:
    """Stable exact-duplicate hash over the aggressive dedup key."""
    return hashlib.sha1(dedup_key(text).encode("utf-8")).hexdigest()


def stable_id(*parts: object) -> str:
    """Deterministic short id for hashing provenance values (e.g. prompts)."""
    h = hashlib.sha1("\x1f".join(str(p) for p in parts).encode("utf-8"))
    return h.hexdigest()[:16]


def word_count(text: str) -> int:
    return len(text.split())


def decode_mixed_lines(raw: bytes) -> tuple[str, dict[str, int]]:
    """Decode a byte stream whose lines use a mix of UTF-8 and Windows-1252.

    Each physical line is tried as UTF-8, then cp1252, then latin-1 (which never
    fails). Counts are returned so the manifest can record what happened.
    """
    stats = {"utf8": 0, "cp1252": 0, "latin1_fallback": 0}
    if raw.startswith(b"\xef\xbb\xbf"):
        raw = raw[3:]
    out = []
    for line in raw.split(b"\n"):
        try:
            out.append(line.decode("utf-8"))
            stats["utf8"] += 1
            continue
        except UnicodeDecodeError:
            pass
        try:
            out.append(line.decode("cp1252"))
            stats["cp1252"] += 1
        except UnicodeDecodeError:
            out.append(line.decode("latin-1"))
            stats["latin1_fallback"] += 1
    return "\n".join(out), stats


def unique_trigram_ratio(text: str, min_trigrams: int = 20) -> float:
    """Unique word trigrams / all trigrams (1.0 for texts too short to judge)."""
    w = text.lower().split()
    grams = list(zip(w, w[1:], w[2:]))
    return len(set(grams)) / len(grams) if len(grams) >= min_trigrams else 1.0


### `ml/scriptsense_ml/data/label_mapping.py`
Explicit, tested mappings from each source dataset's labels to the ScriptSense taxonomy.

In [ ]:
%%writefile ml/scriptsense_ml/data/label_mapping.py
"""Explicit, tested mappings from each source dataset's labels to the ScriptSense taxonomy.

A mapping returns ``(class_4way, label_binary, provenance_note)``. When a source
label cannot validly support the four-way taxonomy it returns ``UNKNOWN`` for
that head instead of guessing; the row then only trains the head it supports.
Unrecognised labels raise, so a schema change in a dataset fails loudly.
"""
from __future__ import annotations

import re
import unicodedata

from .schema import UNKNOWN, Class4, binary_from_class4

Mapping = tuple[int, int, str]


def _canon(label: str) -> str:
    label = unicodedata.normalize("NFKC", str(label))  # NFKC turns NBSP into a space
    return re.sub(r"\s+", " ", label).strip().casefold()


# --------------------------------------------------------------------------- SemEval
def map_semeval_a(label: int) -> Mapping:
    """SemEval-2024 Task 8 Subtask A: 0 = human-written, 1 = machine-generated.

    Subtask A machine texts are whole-document generations; the task description
    reserves refined/mixed text for Subtask C, so AI_GENERATED is justified.
    """
    if label == 0:
        return Class4.HUMAN_ONLY, 0, "semeval_a: human-written (official label 0)"
    if label == 1:
        return Class4.AI_GENERATED, 1, "semeval_a: machine-generated full text (official label 1)"
    raise ValueError(f"unknown SemEval Subtask A label {label!r}")


# --------------------------------------------------------------------------- RAID
RAID_CHAR_LEVEL_ATTACKS = {
    "homoglyph", "number", "article_deletion", "insert_paragraphs",
    "perplexity_misspelling", "upper_lower", "whitespace", "zero_width_space",
    "alternative_spelling",
}
RAID_SEMANTIC_PARAPHRASE = "paraphrase"   # DIPPER paraphraser: rewrites wording/structure
RAID_WORD_SUBSTITUTION = "synonym"        # model-chosen word swaps; neither clean nor a paraphrase
RAID_KNOWN_ATTACKS = RAID_CHAR_LEVEL_ATTACKS | {RAID_SEMANTIC_PARAPHRASE, RAID_WORD_SUBSTITUTION, "none"}


def map_raid(model: str, attack: str) -> Mapping:
    model, attack = str(model).strip().lower(), str(attack).strip().lower()
    if attack not in RAID_KNOWN_ATTACKS:
        raise ValueError(f"unknown RAID attack {attack!r}")
    human = model == "human"

    if attack == "none":
        if human:
            return Class4.HUMAN_ONLY, 0, "raid: human source text, no attack"
        return Class4.AI_GENERATED, 1, f"raid: clean {model} generation"

    if attack == RAID_SEMANTIC_PARAPHRASE:
        if human:
            # Human text rewritten by a paraphrase model = human text revised by AI.
            return Class4.MIXED_OR_AI_ASSISTED, 1, "raid: human text paraphrased by DIPPER"
        return Class4.AI_PARAPHRASED_OR_REFINED, 1, f"raid: {model} generation paraphrased by DIPPER"

    if attack == RAID_WORD_SUBSTITUTION:
        # Word-level model substitution: AI involvement is certain for AI rows, but
        # it is not documented as a semantic rewrite -> four-way label unknown.
        if human:
            return UNKNOWN, UNKNOWN, "raid: human text with synonym substitution (label unsupported)"
        return UNKNOWN, 1, f"raid: {model} generation with synonym substitution"

    # Character/format-level perturbations do not change authorship; never "humanized".
    if human:
        return Class4.HUMAN_ONLY, 0, f"raid: human text with {attack} perturbation"
    return Class4.AI_GENERATED, 1, f"raid: {model} generation with {attack} perturbation"


# --------------------------------------------------------------------------- TXD-22
TXD22_AI_SYSTEMS = {
    "chatgpt", "gemini", "claude", "llama", "mistral ai", "grok", "copilot",
    "perplexity", "deepseek", "deepai", "qwen", "gemma", "poe", "nova", "pi",
    "z.ai", "blackboxai",
}
TXD22_SPECIAL = {
    "human written": (Class4.HUMAN_ONLY, "txd22: human-written, no AI assistance (consented participants)"),
    "ai-generated and ai-refined": (
        Class4.AI_PARAPHRASED_OR_REFINED, "txd22: AI-generated text subsequently AI-refined"),
    "ai-generated and human-written": (
        Class4.MIXED_OR_AI_ASSISTED, "txd22: mixture of AI-generated and human-written content"),
    "human-written and ai-refined": (
        Class4.MIXED_OR_AI_ASSISTED, "txd22: human-written text refined by AI"),
    "ai-generated and ai-refined and human-written": (
        Class4.MIXED_OR_AI_ASSISTED, "txd22: mixture of AI-generated, AI-refined and human-written content"),
}


def map_txd22(source: str) -> Mapping:
    key = _canon(source)
    if key in TXD22_AI_SYSTEMS:
        return Class4.AI_GENERATED, 1, f"txd22: generated by {source.strip()}, not manually rewritten"
    if key in TXD22_SPECIAL:
        c4, note = TXD22_SPECIAL[key]
        return c4, binary_from_class4(c4), note
    raise ValueError(f"unknown TXD-22 Source label {source!r}")


def txd22_generator(source: str) -> str:
    key = _canon(source)
    if key in TXD22_AI_SYSTEMS:
        return key
    return "human" if key == "human written" else "mixed:" + key


### `ml/scriptsense_ml/data/deduplication.py`
Exact and near-duplicate detection.

In [ ]:
%%writefile ml/scriptsense_ml/data/deduplication.py
"""Exact and near-duplicate detection.

Exact duplicates: identical ``text_hash`` (aggressive normalization).
Near duplicates: MinHash over word 5-gram shingles + LSH banding, then candidate
pairs are confirmed by estimated Jaccard >= ``threshold``.

Everything here is deterministic: word hashes use CRC32 (not Python's salted
``hash``) and permutation parameters come from a seeded generator.
"""
from __future__ import annotations

import zlib
from collections import defaultdict
from dataclasses import dataclass

import numpy as np

from .normalization import dedup_key

_MASK64 = np.uint64(0xFFFFFFFFFFFFFFFF)


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))

    def find(self, x: int) -> int:
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a: int, b: int) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)


def shingle_hashes(text: str, n: int = 5) -> np.ndarray:
    words = dedup_key(text).split()
    if not words:
        return np.zeros(1, dtype=np.uint64)
    wh = np.fromiter((zlib.crc32(w.encode("utf-8")) for w in words), dtype=np.uint64, count=len(words))
    if len(words) < n:
        n = len(words)
    # Polynomial combination of n consecutive word hashes (wraps mod 2^64).
    acc = np.zeros(len(words) - n + 1, dtype=np.uint64)
    base = np.uint64(1000003)
    with np.errstate(over="ignore"):
        for i in range(n):
            acc = acc * base + wh[i: len(words) - n + 1 + i]
    return np.unique(acc)


@dataclass
class MinHasher:
    num_perm: int = 128
    seed: int = 1234

    def __post_init__(self):
        rng = np.random.default_rng(self.seed)
        self.a = rng.integers(1, 2**63, size=self.num_perm, dtype=np.uint64) | np.uint64(1)
        self.b = rng.integers(0, 2**63, size=self.num_perm, dtype=np.uint64)

    def signature(self, text: str) -> np.ndarray:
        sh = shingle_hashes(text)
        with np.errstate(over="ignore"):
            # multiply-shift universal hashing, one column per permutation
            h = (sh[:, None] * self.a[None, :] + self.b[None, :]) >> np.uint64(32)
        return h.min(axis=0).astype(np.uint32)


def exact_duplicate_groups(hashes: list[str]) -> dict[str, list[int]]:
    groups: dict[str, list[int]] = defaultdict(list)
    for i, h in enumerate(hashes):
        groups[h].append(i)
    return {h: idx for h, idx in groups.items() if len(idx) > 1}


def near_duplicate_pairs(
    signatures: np.ndarray, bands: int = 16, threshold: float = 0.8
) -> list[tuple[int, int, float]]:
    """Return (i, j, estimated_jaccard) for i<j whose signatures collide in an LSH band
    and whose estimated Jaccard similarity is at least ``threshold``."""
    n, num_perm = signatures.shape
    if num_perm % bands:
        raise ValueError("num_perm must be divisible by bands")
    rows = num_perm // bands
    seen: set[tuple[int, int]] = set()
    out = []
    for b in range(bands):
        buckets: dict[bytes, list[int]] = defaultdict(list)
        block = np.ascontiguousarray(signatures[:, b * rows:(b + 1) * rows])
        for i in range(n):
            buckets[block[i].tobytes()].append(i)
        for members in buckets.values():
            if len(members) < 2 or len(members) > 5000:  # giant buckets = degenerate (e.g. empty) texts
                continue
            for x in range(len(members)):
                for y in range(x + 1, len(members)):
                    pair = (members[x], members[y])
                    if pair in seen:
                        continue
                    seen.add(pair)
                    j = float(np.mean(signatures[pair[0]] == signatures[pair[1]]))
                    if j >= threshold:
                        out.append((pair[0], pair[1], j))
    return out


### `ml/scriptsense_ml/data/splitting.py`
Lineage-grouped split assignment and leakage auditing.

In [ ]:
%%writefile ml/scriptsense_ml/data/splitting.py
"""Lineage-grouped split assignment and leakage auditing.

Rows are connected when they share any group key (prompt, source document,
generation lineage, exact-duplicate hash, near-duplicate cluster). Each connected
component is assigned to exactly one split, so variants of one original never
straddle train and evaluation data.

Publisher-fixed splits (e.g. the SemEval official dev set) pin their component.
If a component contains rows pinned to different evaluation splits, it cannot be
assigned without leakage and is reported as a conflict for removal.
"""
from __future__ import annotations

import hashlib
from collections import Counter, defaultdict

from .deduplication import UnionFind


def connected_components(group_keys: list[list[str]]) -> list[int]:
    """Return a component id (smallest row index in the component) per row."""
    uf = UnionFind(len(group_keys))
    first_seen: dict[str, int] = {}
    for i, keys in enumerate(group_keys):
        for k in keys:
            if not k:
                continue
            if k in first_seen:
                uf.union(first_seen[k], i)
            else:
                first_seen[k] = i
    return [uf.find(i) for i in range(len(group_keys))]


def _unit_interval(key: str, seed: int) -> float:
    h = hashlib.sha256(f"{seed}:{key}".encode()).digest()
    return int.from_bytes(h[:8], "big") / 2**64


def assign_splits(
    components: list[int],
    component_names: dict[int, str],
    fixed_splits: list[str],
    ratios: dict[str, float],
    seed: int,
) -> tuple[list[str], dict[int, list[str]]]:
    """Assign one split per component.

    ``fixed_splits[i]`` is the publisher split for row i or "" if free.
    ``ratios`` maps split name -> probability for free components.
    ``component_names`` gives a stable, order-independent name per component
    (e.g. its smallest row_id) so assignment is reproducible across runs.
    Returns per-row splits ("" for conflicting components) and the conflicts.
    """
    total = sum(ratios.values())
    cumulative, acc = [], 0.0
    for name, r in ratios.items():
        acc += r / total
        cumulative.append((acc, name))

    pins: dict[int, set[str]] = defaultdict(set)
    for comp, fixed in zip(components, fixed_splits):
        if fixed:
            pins[comp].add(fixed)

    decided: dict[int, str] = {}
    conflicts: dict[int, list[str]] = {}
    for comp in set(components):
        pinned = pins.get(comp, set())
        eval_pins = pinned - {"train"}
        if len(eval_pins) > 1:
            conflicts[comp] = sorted(pinned)
            decided[comp] = ""
        elif eval_pins:
            decided[comp] = next(iter(eval_pins))  # evaluation pin wins; train rows in it get moved/dropped
        elif pinned:
            decided[comp] = "train"
        else:
            u = _unit_interval(component_names[comp], seed)
            decided[comp] = next(name for edge, name in cumulative if u < edge or edge == cumulative[-1][0])
    return [decided[c] for c in components], conflicts


def leakage_report(splits: list[str], group_keys: list[list[str]]) -> dict:
    """Every group key must live in exactly one split. Returns offending keys."""
    key_splits: dict[str, set[str]] = defaultdict(set)
    for s, keys in zip(splits, group_keys):
        if not s:
            continue
        for k in keys:
            if k:
                key_splits[k].add(s)
    leaks = {k: sorted(v) for k, v in key_splits.items() if len(v) > 1}
    kinds = Counter(k.split(":", 2)[0] + ":" + k.split(":", 2)[1] if k.count(":") >= 2 else k for k in leaks)
    return {"num_leaking_keys": len(leaks), "by_key_type": dict(kinds), "examples": dict(list(leaks.items())[:20])}


class LeakageError(RuntimeError):
    pass


def assert_no_leakage(splits: list[str], group_keys: list[list[str]]) -> None:
    rep = leakage_report(splits, group_keys)
    if rep["num_leaking_keys"]:
        raise LeakageError(f"split leakage detected: {rep}")


### `ml/scriptsense_ml/data/io.py`
Read/write canonical row tables with schema validation and content hashes.

In [ ]:
%%writefile ml/scriptsense_ml/data/io.py
"""Read/write canonical row tables with schema validation and content hashes."""
from __future__ import annotations

import hashlib
from pathlib import Path

import pandas as pd

from .schema import COLUMNS, validate_row

PREPROCESSING_VERSION = "1.0.0"


def rows_to_frame(rows: list[dict]) -> pd.DataFrame:
    for r in rows:
        validate_row(r)
    df = pd.DataFrame(rows, columns=COLUMNS)
    dup = df["row_id"].duplicated()
    if dup.any():
        raise ValueError(f"duplicate row_id values, e.g. {df.loc[dup, 'row_id'].head(3).tolist()}")
    return df


def write_table(df: pd.DataFrame, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    return file_sha256(path)


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()


### `ml/scriptsense_ml/data/sources.py`
Readers that turn each raw dataset into canonical ScriptSense rows (see schema.COLUMNS).

In [ ]:
%%writefile ml/scriptsense_ml/data/sources.py
"""Readers that turn each raw dataset into canonical ScriptSense rows (see schema.COLUMNS)."""
from __future__ import annotations

import io
import json
import zipfile
from pathlib import Path
from typing import Iterator

import pandas as pd

from . import label_mapping as lm
from .normalization import decode_mixed_lines, dedup_key, normalize_for_model, stable_id

TXD22_LICENSE = "CC BY 4.0 - Iqbal & Kashem, TXD-22, Mendeley Data v1, doi:10.17632/prcjcggtjf.1"
SEMEVAL_LICENSE = "SemEval-2024 Task 8 (Zenodo 11002158); license unstated in record - research use only"
RAID_LICENSE = "RAID (Dugan et al., ACL 2024), HF liamdugan/raid, MIT per dataset card"


def _row(**kw) -> dict:
    raw = kw.pop("raw_text")
    base = {
        "row_id": f"{kw['source_dataset']}:{kw['source_row_id']}",
        "raw_text": raw,
        "text": normalize_for_model(raw),
        "original_document_id": "",
        "prompt_id": "",
        "generator": "",
        "domain": "",
        "attack_or_edit_type": "none",
        "split": "",
        "split_fixed": False,
        "group_keys": "",
    }
    base.update(kw)
    return base


# --------------------------------------------------------------------------- TXD-22
def read_txd22(path: Path) -> tuple[list[dict], dict]:
    text, decode_stats = decode_mixed_lines(Path(path).read_bytes())
    df = pd.read_csv(io.StringIO(text), dtype=str, keep_default_na=False)
    expected = ["Question", "Answer", "Source"]
    if list(df.columns) != expected:
        raise ValueError(f"TXD-22 schema changed: {list(df.columns)} != {expected}")
    rows = []
    for idx, rec in enumerate(df.itertuples(index=False)):
        c4, b, note = lm.map_txd22(rec.Source)
        # Normalised question text defines the prompt group; all 22 answers to it share a split.
        prompt_id = stable_id("txd22", dedup_key(rec.Question))
        gen = lm.txd22_generator(rec.Source)
        rows.append(_row(
            source_dataset="txd22",
            source_row_id=str(idx),
            raw_text=rec.Answer,
            prompt_id=prompt_id,
            generator=gen,
            attack_or_edit_type=gen.removeprefix("mixed:") if gen.startswith("mixed:") else "none",
            provenance=note,
            source_label=rec.Source,
            class_4way=int(c4),
            label_binary=int(b),
            group_keys=f"txd22:prompt:{prompt_id}",
            license_reference=TXD22_LICENSE,
        ))
    return rows, {"decode_line_stats": decode_stats, "raw_rows": len(df)}


# --------------------------------------------------------------------------- SemEval
SEMEVAL_A_FILES = {
    # member-name suffix -> (publisher split, our split)
    "subtaskA_train_monolingual.jsonl": "train",
    "subtaskA_dev_monolingual.jsonl": "validation",
    "subtaskA_test_monolingual.jsonl": "test",  # gold-labelled official test set
}


def iter_semeval_a(zip_path: Path) -> Iterator[tuple[str, dict]]:
    """Yield (our_split, raw_json_record) for Subtask A monolingual files in the zip."""
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.namelist():
            name = member.rsplit("/", 1)[-1]
            if name.startswith("._") or name not in SEMEVAL_A_FILES:
                continue
            split = SEMEVAL_A_FILES[name]
            with zf.open(member) as fh:
                for line in io.TextIOWrapper(fh, encoding="utf-8"):
                    if line.strip():
                        yield split, json.loads(line)


def semeval_a_row(split: str, rec: dict) -> dict:
    c4, b, note = lm.map_semeval_a(int(rec["label"]))
    rid = f"{split}-{rec['id']}"
    return _row(
        source_dataset="semeval_a",
        source_row_id=rid,
        raw_text=rec["text"],
        generator=str(rec.get("model", "")),
        domain=str(rec.get("source", "")),
        provenance=note,
        source_label=str(rec["label"]),
        class_4way=int(c4),
        label_binary=int(b),
        split=split,
        split_fixed=True,
        group_keys=f"semeval_a:doc:{rid}",
        license_reference=SEMEVAL_LICENSE,
    )


# --------------------------------------------------------------------------- RAID
def raid_row(rec: dict, line_no: int) -> dict | None:
    c4, b, note = lm.map_raid(rec["model"], rec["attack"])
    if c4 == lm.UNKNOWN and b == lm.UNKNOWN:
        return None
    lineage = rec["source_id"]
    return _row(
        source_dataset="raid",
        source_row_id=str(rec["id"]) or f"line{line_no}",
        raw_text=rec["generation"],
        original_document_id=lineage,
        prompt_id=stable_id("raid", rec.get("title", "")),
        generator=str(rec["model"]),
        domain=str(rec["domain"]),
        attack_or_edit_type=str(rec["attack"]),
        provenance=note + f"; decoding={rec.get('decoding') or 'n/a'}, rep_penalty={rec.get('repetition_penalty') or 'n/a'}",
        source_label=f"model={rec['model']};attack={rec['attack']}",
        class_4way=int(c4),
        label_binary=int(b),
        group_keys=f"raid:lineage:{lineage}",
        license_reference=RAID_LICENSE,
    )


## 2. Code: Package: model, training, evaluation

### `ml/scriptsense_ml/models/multitask.py`
Encoder with a four-way provenance head and an auxiliary binary AI-involvement head.

In [ ]:
%%writefile ml/scriptsense_ml/models/multitask.py
"""Encoder with a four-way provenance head and an auxiliary binary AI-involvement head.

Works with any Hugging Face encoder (microsoft/deberta-v3-base primary, roberta-base baseline).
Labels equal to -1 (UNKNOWN) are ignored by the corresponding loss, so a row that only
supports the binary label never pushes the four-way head toward a guessed class.
"""
from __future__ import annotations

import json
from pathlib import Path

import torch
from torch import nn
from transformers import AutoConfig, AutoModel

from ..data.schema import BINARY_NAMES, CLASS4_NAMES, UNKNOWN


class MultiTaskDetector(nn.Module):
    def __init__(self, encoder_name: str, dropout: float = 0.1, encoder_config=None):
        super().__init__()
        if encoder_config is not None:  # tests: tiny randomly initialised encoder
            self.encoder = AutoModel.from_config(encoder_config)
        else:
            # fp32 master weights: transformers>=5 otherwise keeps the checkpoint dtype, and the
            # deberta-v3-base checkpoint is stored in fp16, which diverges to NaN when trained.
            self.encoder = AutoModel.from_pretrained(encoder_name, dtype=torch.float32)
        hidden = self.encoder.config.hidden_size
        self.encoder_name = encoder_name
        self.dropout = nn.Dropout(dropout)
        self.head4 = nn.Linear(hidden, len(CLASS4_NAMES))
        self.head2 = nn.Linear(hidden, len(BINARY_NAMES))

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0])
        return self.head4(cls), self.head2(cls)

    def save(self, path: Path, metadata: dict) -> None:
        path.mkdir(parents=True, exist_ok=True)
        self.encoder.save_pretrained(path / "encoder")
        torch.save({"head4": self.head4.state_dict(), "head2": self.head2.state_dict()}, path / "heads.pt")
        meta = {**metadata, "encoder_name": self.encoder_name,
                "label_map_4way": dict(enumerate(CLASS4_NAMES)), "label_map_binary": dict(enumerate(BINARY_NAMES))}
        (path / "model_meta.json").write_text(json.dumps(meta, indent=2, default=str))

    @classmethod
    def load(cls, path: Path, map_location="cpu") -> "MultiTaskDetector":
        meta = json.loads((path / "model_meta.json").read_text())
        model = cls.__new__(cls)
        nn.Module.__init__(model)
        model.encoder = AutoModel.from_pretrained(path / "encoder", dtype=torch.float32)
        model.encoder_name = meta["encoder_name"]
        hidden = model.encoder.config.hidden_size
        model.dropout = nn.Dropout(0.0)
        model.head4 = nn.Linear(hidden, len(CLASS4_NAMES))
        model.head2 = nn.Linear(hidden, len(BINARY_NAMES))
        heads = torch.load(path / "heads.pt", map_location=map_location, weights_only=True)
        model.head4.load_state_dict(heads["head4"])
        model.head2.load_state_dict(heads["head2"])
        return model


def multitask_loss(logits4, logits2, y4, y2, class_weights4=None, binary_weight: float = 0.5,
                   four_way_weight: float = 1.0) -> tuple[torch.Tensor, dict]:
    """Weighted sum of masked cross-entropies. ``four_way_weight=0`` gives a binary-only baseline."""
    ce4 = nn.CrossEntropyLoss(weight=class_weights4, ignore_index=UNKNOWN, reduction="sum")
    ce2 = nn.CrossEntropyLoss(ignore_index=UNKNOWN, reduction="sum")
    n4, n2 = (y4 != UNKNOWN).sum(), (y2 != UNKNOWN).sum()
    # Normalise by the (weighted) count of supervised rows, guarding all-unknown batches.
    if class_weights4 is not None:
        denom4 = class_weights4[y4[y4 != UNKNOWN]].sum().clamp(min=1e-8)
    else:
        denom4 = n4.clamp(min=1)
    l4 = ce4(logits4, y4) / denom4 if n4 > 0 else logits4.sum() * 0.0
    l2 = ce2(logits2, y2) / n2.clamp(min=1) if n2 > 0 else logits2.sum() * 0.0
    total = four_way_weight * l4 + binary_weight * l2
    return total, {"loss_4way": float(l4.detach()), "loss_binary": float(l2.detach())}


def class_weights_from_counts(counts: dict[int, int], power: float = 0.5) -> torch.Tensor:
    """Inverse-frequency weights tempered by ``power`` (0.5 = inverse sqrt), normalised to mean 1
    over present classes. Absent classes get weight 0."""
    w = torch.zeros(len(CLASS4_NAMES))
    for c, n in counts.items():
        if c != UNKNOWN and n > 0:
            w[c] = float(n) ** -power
    present = w > 0
    w[present] = w[present] / w[present].mean()
    return w


### `ml/scriptsense_ml/training/data.py`
Training data selection: split filtering, ablation filters, documented subsets.

In [ ]:
%%writefile ml/scriptsense_ml/training/data.py
"""Training data selection: split filtering, ablation filters, documented subsets."""
from __future__ import annotations

import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

from ..data.schema import Class4

COLS = ["row_id", "text", "class_4way", "label_binary", "split", "source_dataset", "generator",
        "domain", "attack_or_edit_type", "prompt_id", "original_document_id"]


def load_dataset(path: Path, columns=COLS) -> pd.DataFrame:
    return pd.read_parquet(path, columns=columns)


def apply_ablation(train: pd.DataFrame, ablation: dict) -> tuple[pd.DataFrame, dict]:
    """Ablations only ever remove rows from the TRAIN split, so every run is compared on the
    same validation/test data."""
    removed = {}
    keep = pd.Series(True, index=train.index)
    if ablation.get("exclude_raid_paraphrase"):
        m = (train.source_dataset == "raid") & (train.attack_or_edit_type == "paraphrase")
        removed["raid_paraphrase"] = int(m.sum())
        keep &= ~m
    if ablation.get("exclude_txd22_mixed"):
        m = (train.source_dataset == "txd22") & train.class_4way.isin(
            [Class4.AI_PARAPHRASED_OR_REFINED, Class4.MIXED_OR_AI_ASSISTED])
        removed["txd22_mixed_or_refined"] = int(m.sum())
        keep &= ~m
    for src in ablation.get("exclude_sources", []) or []:
        m = train.source_dataset == src
        removed[f"source:{src}"] = int(m.sum())
        keep &= ~m
    return train[keep], removed


def stratified_subset(df: pd.DataFrame, n: int | None, seed: int) -> pd.DataFrame:
    """Deterministic subset keeping the (source, class) mix proportional. ``None`` = all rows."""
    if n is None or n >= len(df):
        return df
    strata = df.source_dataset + "/" + df.class_4way.astype(str) + "/" + df.label_binary.astype(str)
    frac = n / len(df)
    parts = [g.sample(n=max(1, int(round(len(g) * frac))), random_state=seed) for _, g in df.groupby(strata)]
    return pd.concat(parts).sort_values("row_id")


def row_ids_digest(df: pd.DataFrame) -> str:
    return hashlib.sha256("\n".join(sorted(df.row_id)).encode()).hexdigest()


class TextDataset:
    def __init__(self, df: pd.DataFrame):
        self.texts = df.text.tolist()
        self.y4 = df.class_4way.to_numpy(np.int64)
        self.y2 = df.label_binary.to_numpy(np.int64)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i], self.y4[i], self.y2[i]


### `ml/scriptsense_ml/training/trainer.py`
Training loop for the multi-task detector.

In [ ]:
%%writefile ml/scriptsense_ml/training/trainer.py
"""Training loop for the multi-task detector.

Selection uses validation only. Final test/challenge data are never loaded here.
"""
from __future__ import annotations

import json
import math
import os
import platform
import random
import subprocess
import time
from collections import Counter
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from ..data.schema import UNKNOWN
from ..evaluation.metrics import binary_metrics, four_class_metrics
from ..inference.predict import native_bf16_supported, predict_logits
from ..models.multitask import MultiTaskDetector, class_weights_from_counts, multitask_loss
from .data import TextDataset

PACKAGES = ["torch", "transformers", "tokenizers", "sentencepiece", "numpy", "pandas", "scikit-learn", "scipy"]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def resolve_precision(precision: str, device: str):
    if not device.startswith("cuda") or precision == "fp32":
        return None, False
    # torch.cuda.is_bf16_supported() also counts *emulated* bf16 (e.g. a Colab T4), which is very
    # slow, so "auto" only picks bf16 on GPUs with native support (compute capability >= 8).
    if precision == "bf16" or (precision == "auto" and native_bf16_supported()):
        return torch.bfloat16, False
    return torch.float16, True  # fp16 needs a GradScaler


def environment_info() -> dict:
    versions = {}
    for p in PACKAGES:
        try:
            versions[p] = importlib_metadata.version(p)
        except importlib_metadata.PackageNotFoundError:
            versions[p] = None
    try:
        commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True,
                                cwd=Path(__file__).parent).stdout.strip()
    except OSError:
        commit = None
    return {"python": platform.python_version(), "packages": versions, "git_commit": commit,
            "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}


def evaluate(model, tokenizer, df, cfg, device, amp_dtype) -> tuple[dict, np.ndarray, np.ndarray]:
    l4, l2 = predict_logits(model, tokenizer, df.text.tolist(), cfg["max_length"], cfg["eval_batch_size"],
                            device, amp_dtype)
    p4 = torch.softmax(torch.from_numpy(l4), 1).numpy()
    p2 = torch.softmax(torch.from_numpy(l2), 1).numpy()[:, 1]
    y4, y2 = df.class_4way.to_numpy(), df.label_binary.to_numpy()
    res = {"overall": {"four_way": four_class_metrics(p4, y4), "binary": binary_metrics(p2, y2)}}
    for src, idx in df.groupby("source_dataset").indices.items():
        res[src] = {"four_way": four_class_metrics(p4[idx], y4[idx]), "binary": binary_metrics(p2[idx], y2[idx])}
    for blk in res.values():  # keep the training log compact
        blk["four_way"].pop("reliability_bins", None)
        blk["binary"].pop("reliability_bins", None)
    return res, l4, l2


def selection_score(res: dict, cfg: dict) -> float:
    metric = cfg["selection_metric"]
    if metric == "auto":
        metric = "macro_f1_4way" if cfg["mode"] == "multitask" else "auroc"
    if metric == "macro_f1_4way":
        return res["overall"]["four_way"].get("macro_f1_present_classes", float("nan"))
    return res["overall"]["binary"].get(metric, float("nan"))


# Config keys that must match for a checkpoint to be resumed.
RESUME_KEYS = ["model_name", "mode", "seed", "max_length", "learning_rate", "epochs", "per_device_batch_size",
               "grad_accum_steps", "weight_decay", "warmup_ratio", "binary_loss_weight", "class_weight_power",
               "freeze_word_embeddings", "ablation", "max_train_rows", "max_val_rows"]
MAX_NONFINITE_STEPS = 20


def _rng_state() -> dict:
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def _set_rng_state(st: dict) -> None:
    random.setstate(st["python"])
    np.random.set_state(st["numpy"])
    torch.set_rng_state(st["torch"])
    if st.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(st["cuda"])


def _atomic_save(obj, path: Path) -> None:
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)  # a disconnect mid-write never corrupts the previous checkpoint


def load_tokenizer_cached(name: str, cache_root: Path):
    """DeBERTa-v3 converts its sentencepiece model to a fast tokenizer on every load (~1 min);
    save the converted tokenizer once and reuse it."""
    cache = cache_root / name.replace("/", "__")
    if (cache / "tokenizer_config.json").exists():
        return AutoTokenizer.from_pretrained(cache)
    tok = AutoTokenizer.from_pretrained(name)
    tok.save_pretrained(cache)
    return tok


def epoch_order(n: int, seed: int, epoch: int) -> list[int]:
    """Deterministic shuffle per epoch, so a resumed run sees exactly the same batches."""
    return torch.randperm(n, generator=torch.Generator().manual_seed(seed * 1000 + epoch)).tolist()


def train(cfg: dict, train_df, val_df, run_dir: Path, data_provenance: dict, encoder_config=None) -> dict:
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = run_dir / "last_checkpoint.pt"
    meta_path = run_dir / "run_meta.json"
    resume = cfg.get("resume", True) and ckpt_path.exists()
    if not resume and meta_path.exists() and json.loads(meta_path.read_text()).get("finished_at"):
        raise SystemExit(f"run '{cfg['run_name']}' already finished in {run_dir}; use a new run_name")

    set_seed(cfg["seed"])
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp_dtype, use_scaler = resolve_precision(cfg["precision"], device)

    tokenizer = load_tokenizer_cached(cfg.get("tokenizer_name") or cfg["model_name"], run_dir.parent / "_tokenizer_cache")
    model = MultiTaskDetector(cfg["model_name"], encoder_config=encoder_config).to(device)
    if cfg.get("gradient_checkpointing") and device == "cuda":
        model.encoder.gradient_checkpointing_enable()
    if cfg.get("freeze_word_embeddings"):
        model.encoder.get_input_embeddings().requires_grad_(False)

    four_way_weight = 1.0 if cfg["mode"] == "multitask" else 0.0
    counts = Counter(int(c) for c in train_df.class_4way if c != UNKNOWN)
    cw = class_weights_from_counts(counts, cfg["class_weight_power"]).to(device) if four_way_weight else None

    ds = TextDataset(train_df)
    bs, accum = cfg["per_device_batch_size"], cfg["grad_accum_steps"]
    batches_per_epoch = math.ceil(len(ds) / bs)
    workers = 0 if os.name == "nt" else int(cfg.get("dataloader_workers", 0))

    def collate(batch):
        texts, y4, y2 = zip(*batch)
        enc = tokenizer(list(texts), truncation=True, max_length=cfg["max_length"], padding=True, return_tensors="pt")
        return enc, torch.tensor(y4), torch.tensor(y2)

    no_decay = ("bias", "LayerNorm.weight", "layernorm", "layer_norm")
    params = [
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and not any(k in n for k in no_decay)],
         "weight_decay": cfg["weight_decay"]},
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and any(k in n for k in no_decay)],
         "weight_decay": 0.0},
    ]
    opt = torch.optim.AdamW(params, lr=cfg["learning_rate"])
    steps_per_epoch = math.ceil(batches_per_epoch / accum)
    total_steps = steps_per_epoch * cfg["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(cfg["warmup_ratio"] * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    state = {"epoch": 0, "batch": 0, "step": 0, "best": None, "best_score": -float("inf"),
             "bad_evals": 0, "stop": False, "elapsed_s": 0.0}
    if resume:
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if ck["train_row_ids_sha256"] != data_provenance.get("train_row_ids_sha256"):
            raise SystemExit("checkpoint was trained on different rows; rebuild the same dataset or use a new run_name")
        changed = [k for k in RESUME_KEYS if ck["config"].get(k) != cfg.get(k)]
        if changed:
            raise SystemExit(f"checkpoint config differs on {changed}; use the original settings or a new run_name")
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"])
        scaler.load_state_dict(ck["scaler"])
        state = ck["state"]
        _set_rng_state(ck["rng"])
        meta = json.loads(meta_path.read_text())
        meta.setdefault("resumed_at", []).append({"at": datetime.now(timezone.utc).isoformat(), "step": state["step"]})
        print(f"[resume] epoch={state['epoch']} batch={state['batch']}/{batches_per_epoch} "
              f"step={state['step']}/{total_steps}", flush=True)
    else:
        meta = {
            "config": cfg, "environment": environment_info(), "device": device,
            "precision": str(amp_dtype) if amp_dtype else "fp32",
            "started_at": datetime.now(timezone.utc).isoformat(), "data": data_provenance,
            "train_class_counts_4way": {int(k): v for k, v in sorted(counts.items())},
            "train_binary_counts": {int(k): int(v) for k, v in train_df.label_binary.value_counts().items()},
            "effective_class_weights_4way": cw.tolist() if cw is not None else None,
            "optimizer_steps_planned": total_steps,
            "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
            "total_params": sum(p.numel() for p in model.parameters()),
        }
    meta_path.write_text(json.dumps(meta, indent=2, default=str))
    log = open(run_dir / "train_log.jsonl", "a", encoding="utf-8")
    t_start = time.time()

    def elapsed() -> float:
        return round(state["elapsed_s"] + time.time() - t_start, 1)

    def save_checkpoint() -> None:
        _atomic_save({"model": model.state_dict(), "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
                      "scaler": scaler.state_dict(), "state": {**state, "elapsed_s": elapsed()}, "rng": _rng_state(),
                      "config": cfg, "train_row_ids_sha256": data_provenance.get("train_row_ids_sha256")}, ckpt_path)

    def do_eval(epoch):
        res, l4, l2 = evaluate(model, tokenizer, val_df, cfg, device, amp_dtype)
        score = selection_score(res, cfg)
        rec = {"event": "eval", "epoch": epoch, "step": state["step"], "selection_score": score,
               "elapsed_s": elapsed(), "validation": res}
        log.write(json.dumps(rec, default=str) + "\n")
        log.flush()
        print(f"[eval] epoch={epoch} step={state['step']} score={score:.4f}", flush=True)
        if score > state["best_score"]:
            state["best_score"], state["bad_evals"] = score, 0
            state["best"] = {"epoch": epoch, "step": state["step"], "score": score}
            model.save(run_dir / "best", {**{k: meta[k] for k in ("config", "environment", "data", "precision")},
                                          "selected_at": state["best"],
                                          "tokenizer_name": cfg.get("tokenizer_name") or cfg["model_name"]})
            tokenizer.save_pretrained(run_dir / "best" / "tokenizer")
            np.savez_compressed(run_dir / "best" / "val_logits.npz", logits4=l4, logits2=l2,
                                y4=val_df.class_4way.to_numpy(), y2=val_df.label_binary.to_numpy(),
                                row_id=val_df.row_id.to_numpy())
        else:
            state["bad_evals"] += 1
            if state["bad_evals"] > cfg["early_stopping_patience"]:
                state["stop"] = True
        model.train()

    eval_every = cfg.get("eval_every_steps")
    ckpt_every = int(cfg.get("checkpoint_every_steps") or 0)
    stop_after = cfg.get("stop_after_steps")  # simulates an interruption (tests / debugging)
    nonfinite = 0

    for epoch in range(state["epoch"], cfg["epochs"]):
        if state["stop"]:
            break
        start = state["batch"] if epoch == state["epoch"] else 0
        order = epoch_order(len(ds), cfg["seed"], epoch)[start * bs:]
        loader = DataLoader(Subset(ds, order), batch_size=bs, shuffle=False, collate_fn=collate, num_workers=workers)
        model.train()
        opt.zero_grad(set_to_none=True)
        for j, (enc, y4, y2) in enumerate(loader):
            i = start + j  # batch index within the epoch
            enc, y4, y2 = enc.to(device), y4.to(device), y2.to(device)
            with torch.autocast(device_type=device, dtype=amp_dtype, enabled=amp_dtype is not None):
                l4, l2 = model(enc["input_ids"], enc["attention_mask"])
            loss, parts = multitask_loss(l4.float(), l2.float(), y4, y2, cw, cfg["binary_loss_weight"], four_way_weight)
            scaler.scale(loss / accum).backward()
            if (i + 1) % accum and i + 1 != batches_per_epoch:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad(set_to_none=True)
            state.update(step=state["step"] + 1, epoch=epoch, batch=i + 1)
            step = state["step"]

            nonfinite = 0 if torch.isfinite(loss) else nonfinite + 1
            if nonfinite >= MAX_NONFINITE_STEPS:
                log.close()
                raise RuntimeError(f"loss non-finite for {nonfinite} consecutive steps at step {step}; training diverged")
            if step % 50 == 0 or step == 1:
                rec = {"event": "train", "epoch": epoch, "step": step, "of_steps": total_steps,
                       "loss": float(loss.detach()), **parts, "lr": sched.get_last_lr()[0], "elapsed_s": elapsed()}
                log.write(json.dumps(rec) + "\n")
                log.flush()
                print(f"[train] {rec}", flush=True)
            if eval_every and step % eval_every == 0:
                do_eval(epoch)
            if ckpt_every and step % ckpt_every == 0:
                save_checkpoint()
            if stop_after and step >= stop_after:
                save_checkpoint()
                log.close()
                return {**meta, "interrupted_at_step": step}
            if state["stop"]:
                break
        if state["stop"]:
            break
        if not eval_every:
            do_eval(epoch)
        state.update(epoch=epoch + 1, batch=0)
        if ckpt_every:
            save_checkpoint()

    log.close()
    meta.update({"finished_at": datetime.now(timezone.utc).isoformat(), "best": state["best"],
                 "optimizer_steps_done": state["step"], "early_stopped": state["stop"], "train_seconds": elapsed()})
    meta_path.write_text(json.dumps(meta, indent=2, default=str))
    if ckpt_path.exists() and not cfg.get("keep_last_checkpoint"):
        ckpt_path.unlink()  # ~2 GB; the selected model lives in best/
    return meta


### `ml/scriptsense_ml/evaluation/metrics.py`
Evaluation metrics for the AI-writing detector (CLAUDE.md section 6).

In [ ]:
%%writefile ml/scriptsense_ml/evaluation/metrics.py
"""Evaluation metrics for the AI-writing detector (CLAUDE.md section 6).

Rows whose label is UNKNOWN for a head are excluded from that head's metrics.
"""
from __future__ import annotations

import numpy as np
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, precision_recall_fscore_support, roc_auc_score,
)

from ..data.schema import CLASS4_NAMES, UNKNOWN

# An FPR operating point is reported as reliable only if it rests on >= this many expected
# false positives (e.g. 1% FPR needs >= 1,000 human samples).
MIN_EXPECTED_FALSE_POSITIVES = 10


def expected_calibration_error(probs: np.ndarray, y: np.ndarray, n_bins: int = 15) -> tuple[float, list[dict]]:
    """Top-label ECE plus the reliability-diagram bins. ``probs`` is (n, k)."""
    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece, bins = 0.0, []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            gap = abs(correct[m].mean() - conf[m].mean())
            ece += m.mean() * gap
            bins.append({"lo": float(lo), "hi": float(hi), "count": int(m.sum()),
                         "accuracy": float(correct[m].mean()), "confidence": float(conf[m].mean())})
    return float(ece), bins


def multiclass_brier(probs: np.ndarray, y: np.ndarray) -> float:
    onehot = np.eye(probs.shape[1])[y]
    return float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))


def four_class_metrics(probs4: np.ndarray, y4: np.ndarray) -> dict:
    m = y4 != UNKNOWN
    p, y = probs4[m], y4[m]
    if len(y) == 0:
        return {"n": 0}
    pred = p.argmax(axis=1)
    labels = list(range(len(CLASS4_NAMES)))
    prec, rec, f1, sup = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    present = [i for i in labels if sup[i] > 0]
    ece, bins = expected_calibration_error(p, y)
    return {
        "n": int(len(y)),
        "macro_f1_present_classes": float(f1_score(y, pred, labels=present, average="macro", zero_division=0)),
        "classes_present": [CLASS4_NAMES[i] for i in present],
        "accuracy": float((pred == y).mean()),
        "per_class": {CLASS4_NAMES[i]: {"precision": float(prec[i]), "recall": float(rec[i]),
                                        "f1": float(f1[i]), "support": int(sup[i])} for i in labels},
        "confusion_matrix": {"labels": CLASS4_NAMES,
                             "matrix": confusion_matrix(y, pred, labels=labels).tolist()},
        "brier": multiclass_brier(p, y),
        "ece": ece,
        "reliability_bins": bins,
    }


def threshold_for_fpr(scores_neg: np.ndarray, target_fpr: float) -> float:
    """Smallest threshold t such that mean(scores_neg >= t) <= target_fpr (predict positive if score >= t)."""
    s = np.sort(scores_neg)[::-1]
    k = int(np.floor(target_fpr * len(s)))  # number of negatives allowed above the threshold
    if k >= len(s):
        return float(s[-1])
    return float(np.nextafter(s[k], np.inf))


def binary_metrics(p_ai: np.ndarray, yb: np.ndarray, threshold: float = 0.5,
                   fpr_points: tuple[float, ...] = (0.01, 0.05)) -> dict:
    m = yb != UNKNOWN
    p, y = p_ai[m], yb[m]
    out: dict = {"n": int(len(y)), "n_human": int((y == 0).sum()), "n_ai": int((y == 1).sum()),
                 "threshold": threshold}
    if len(y) == 0:
        return out
    pred = (p >= threshold).astype(int)
    tp, fp = int(((pred == 1) & (y == 1)).sum()), int(((pred == 1) & (y == 0)).sum())
    fn, tn = int(((pred == 0) & (y == 1)).sum()), int(((pred == 0) & (y == 0)).sum())
    out.update({
        "confusion": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
        "human_false_positive_rate": fp / (fp + tn) if fp + tn else None,
        "ai_recall": tp / (tp + fn) if tp + fn else None,
        "precision": tp / (tp + fp) if tp + fp else None,
        "f1": float(f1_score(y, pred, zero_division=0)),
        "brier": float(np.mean((p - y) ** 2)),
    })
    if len(np.unique(y)) == 2:
        out["auroc"] = float(roc_auc_score(y, p))
        out["auprc"] = float(average_precision_score(y, p))
        ece, bins = expected_calibration_error(np.stack([1 - p, p], axis=1), y)
        out["ece"], out["reliability_bins"] = ece, bins
        # Oracle operating points on THIS set: descriptive only, never used to pick deployment thresholds.
        pts = {}
        for f in fpr_points:
            t = threshold_for_fpr(p[y == 0], f)
            pr = p >= t
            pts[f"fpr_{f}"] = {
                "threshold_on_this_set": t,
                "achieved_fpr": float(pr[y == 0].mean()),
                "recall": float(pr[y == 1].mean()),
                "precision": float((pr & (y == 1)).sum() / max(pr.sum(), 1)),
                "reliable": bool((y == 0).sum() * f >= MIN_EXPECTED_FALSE_POSITIVES),
            }
        out["operating_points_oracle"] = pts
    return out


def recall_by_class_at_threshold(p_ai: np.ndarray, y4: np.ndarray, threshold: float) -> dict:
    """Binary AI-involvement recall broken down by the true four-way class (e.g. paraphrased, mixed)."""
    out = {}
    for i, name in enumerate(CLASS4_NAMES):
        m = y4 == i
        if m.any():
            key = "flag_rate" if i == 0 else "recall"
            out[name] = {key: float((p_ai[m] >= threshold).mean()), "n": int(m.sum())}
    return out


def bootstrap_ci(fn, arrays: tuple[np.ndarray, ...], n_boot: int = 1000, seed: int = 0,
                 groups: np.ndarray | None = None) -> dict:
    """Percentile CI. When ``groups`` is given, resample whole groups (e.g. prompts) not rows."""
    rng = np.random.default_rng(seed)
    n = len(arrays[0])
    vals = []
    if groups is not None:
        uniq, inv = np.unique(groups, return_inverse=True)
        members = [np.flatnonzero(inv == g) for g in range(len(uniq))]
    for _ in range(n_boot):
        if groups is None:
            idx = rng.integers(0, n, n)
        else:
            idx = np.concatenate([members[g] for g in rng.integers(0, len(members), len(members))])
        try:
            vals.append(fn(*(a[idx] for a in arrays)))
        except ValueError:  # e.g. a resample with one class for AUROC
            continue
    vals = np.array(vals, dtype=float)
    if len(vals) == 0:
        return {"low": None, "high": None, "n_boot": 0}
    return {"low": float(np.percentile(vals, 2.5)), "high": float(np.percentile(vals, 97.5)),
            "n_boot": int(len(vals))}


def length_bucket(n_words: np.ndarray) -> np.ndarray:
    return np.select([n_words < 50, n_words < 150, n_words < 400], ["<50", "50-149", "150-399"], ">=400")


### `ml/scriptsense_ml/evaluation/calibration.py`
Temperature scaling (Guo et al., 2017) fit on validation logits only, and FPR-constrained threshold selection on held-out human validation writing.

In [ ]:
%%writefile ml/scriptsense_ml/evaluation/calibration.py
"""Temperature scaling (Guo et al., 2017) fit on validation logits only, and
FPR-constrained threshold selection on held-out human validation writing."""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize_scalar
from scipy.special import log_softmax, softmax

from ..data.schema import UNKNOWN
from .metrics import threshold_for_fpr


def fit_temperature(logits: np.ndarray, y: np.ndarray) -> float:
    m = y != UNKNOWN
    z, t = logits[m].astype(np.float64), y[m]
    if len(t) == 0:
        return 1.0

    def nll(log_t: float) -> float:
        return -log_softmax(z / np.exp(log_t), axis=1)[np.arange(len(t)), t].mean()

    res = minimize_scalar(nll, bounds=(np.log(0.05), np.log(20.0)), method="bounded")
    return float(np.exp(res.x))


def apply_temperature(logits: np.ndarray, temperature: float) -> np.ndarray:
    return softmax(logits.astype(np.float64) / temperature, axis=1)


def select_thresholds(p_ai_val: np.ndarray, yb_val: np.ndarray, target_fprs=(0.01, 0.05)) -> dict:
    """Thresholds on the calibrated binary AI probability that keep the *validation* human
    false-positive rate at or below each target. Reports the support so callers can see when
    an operating point rests on too few human samples."""
    neg = p_ai_val[yb_val == 0]
    pos = p_ai_val[yb_val == 1]
    out = {}
    for f in target_fprs:
        t = threshold_for_fpr(neg, f)
        out[f"fpr_{f}"] = {
            "threshold": t,
            "validation_fpr": float((neg >= t).mean()) if len(neg) else None,
            "validation_recall": float((pos >= t).mean()) if len(pos) else None,
            "n_human_validation": int(len(neg)),
            "expected_false_positives_at_target": float(len(neg) * f),
        }
    return out


## 2. Code: Package: inference

### `ml/scriptsense_ml/inference/predict.py`
Batched logit prediction shared by training-time validation, evaluation and serving.

In [ ]:
%%writefile ml/scriptsense_ml/inference/predict.py
"""Batched logit prediction shared by training-time validation, evaluation and serving."""
from __future__ import annotations

import numpy as np
import torch


def native_bf16_supported() -> bool:
    """True only for GPUs with hardware bf16 (Ampere+). torch.cuda.is_bf16_supported() also
    returns True when bf16 is merely emulated (e.g. Colab T4), which is very slow."""
    return torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8


def inference_amp_dtype(device: str):
    if not device.startswith("cuda"):
        return None
    return torch.bfloat16 if native_bf16_supported() else torch.float16


@torch.no_grad()
def predict_logits(model, tokenizer, texts: list[str], max_length: int, batch_size: int = 32,
                   device: str = "cpu", amp_dtype=None) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    out4, out2 = [], []
    # Sort by length so batches pad little; restore order at the end.
    order = np.argsort([len(t) for t in texts])
    for s in range(0, len(texts), batch_size):
        idx = order[s:s + batch_size]
        enc = tokenizer([texts[i] for i in idx], truncation=True, max_length=max_length,
                        padding=True, return_tensors="pt").to(device)
        with torch.autocast(device_type=device.split(":")[0], dtype=amp_dtype, enabled=amp_dtype is not None):
            l4, l2 = model(enc["input_ids"], enc["attention_mask"])
        out4.append(l4.float().cpu().numpy())
        out2.append(l2.float().cpu().numpy())
    l4 = np.concatenate(out4) if out4 else np.zeros((0, 4))
    l2 = np.concatenate(out2) if out2 else np.zeros((0, 2))
    inv = np.empty_like(order)
    inv[order] = np.arange(len(order))
    return l4[inv], l2[inv]


### `ml/scriptsense_ml/inference/chunking.py`
Eligible-prose detection and overlapping token windows with original character offsets.

In [ ]:
%%writefile ml/scriptsense_ml/inference/chunking.py
"""Eligible-prose detection and overlapping token windows with original character offsets.

Windows are computed on the ORIGINAL extracted text so every chunk maps back to exact
character offsets (and therefore pages). The model sees normalize_for_model(chunk), the
same normalization used in training.
"""
from __future__ import annotations

import re
from dataclasses import dataclass

REFERENCE_HEADINGS = re.compile(
    r"^\s*(references|bibliography|works cited|literature cited|reference list|sources)\s*:?\s*$", re.I)
_PARA_SPLIT = re.compile(r"\n\s*\n")


@dataclass
class Block:
    start: int
    end: int
    eligible: bool
    reason: str = ""


@dataclass
class Window:
    start: int        # character offsets into the original text
    end: int
    n_tokens: int
    new_chars: int    # characters not already covered by the previous window (aggregation weight)


def _alpha_ratio(s: str) -> float:
    chars = [c for c in s if not c.isspace()]
    return sum(c.isalpha() for c in chars) / len(chars) if chars else 0.0


def segment_blocks(text: str, min_words: int = 8) -> list[Block]:
    """Split into paragraphs and mark which are eligible prose.

    Excluded: everything after a references/bibliography heading, short lines (headings,
    captions), and low-alphabetic blocks (tables, formulas, code).
    """
    blocks, pos, in_refs = [], 0, False
    for m in list(_PARA_SPLIT.finditer(text)) + [None]:
        end = m.start() if m else len(text)
        seg = text[pos:end]
        lead = len(seg) - len(seg.lstrip())
        s, e = pos + lead, pos + len(seg.rstrip())
        if e > s:
            body = text[s:e]
            first_line = body.split("\n", 1)[0]
            if REFERENCE_HEADINGS.match(first_line):
                in_refs = True
            if in_refs:
                blocks.append(Block(s, e, False, "references"))
            elif len(body.split()) < min_words:
                blocks.append(Block(s, e, False, "too_short"))
            elif _alpha_ratio(body) < 0.6:
                blocks.append(Block(s, e, False, "non_prose"))
            else:
                blocks.append(Block(s, e, True))
        pos = m.end() if m else len(text)
    return blocks


def eligible_spans(blocks: list[Block]) -> list[tuple[int, int]]:
    """Merge consecutive eligible blocks into spans; windows never cross excluded material."""
    spans: list[list[int]] = []
    prev_eligible = False
    for b in blocks:
        if b.eligible and prev_eligible:
            spans[-1][1] = b.end
        elif b.eligible:
            spans.append([b.start, b.end])
        prev_eligible = b.eligible
    return [(s, e) for s, e in spans]


def token_windows(text: str, span: tuple[int, int], tokenizer, max_tokens: int = 384,
                  overlap: int = 64) -> list[Window]:
    s0, e0 = span
    enc = tokenizer(text[s0:e0], add_special_tokens=False, return_offsets_mapping=True)
    offs = [o for o in enc["offset_mapping"] if o[1] > o[0]]
    if not offs:
        return []
    body = max_tokens - tokenizer.num_special_tokens_to_add()
    stride = max(1, body - overlap)
    windows, covered_to, i = [], s0, 0
    while True:
        chunk = offs[i:i + body]
        ws, we = s0 + chunk[0][0], s0 + chunk[-1][1]
        windows.append(Window(ws, we, len(chunk), max(0, we - max(ws, covered_to))))
        covered_to = max(covered_to, we)
        if i + body >= len(offs):
            break
        i += stride
    return windows


def offset_to_page(offset: int, page_starts: list[int]) -> int:
    """1-based page for a character offset given each page's starting offset."""
    page = 1
    for i, start in enumerate(page_starts):
        if offset >= start:
            page = i + 1
    return page


### `ml/scriptsense_ml/inference/aggregation.py`
Document-level aggregation of calibrated window predictions.

In [ ]:
%%writefile ml/scriptsense_ml/inference/aggregation.py
"""Document-level aggregation of calibrated window predictions.

Definitions (also returned to API clients in ``definitions``):
* document class probabilities: mean of calibrated window four-way probabilities, each window
  weighted by the characters it adds beyond the previous window (so overlap is not double counted).
* ai_signal_score: the same weighted mean of the calibrated binary AI-involvement probability.
* flagged_text_share: share of analysed characters lying in windows whose calibrated binary
  probability is >= the deployed threshold. It is a descriptive statistic of model flags, NOT a
  validated estimate of how much of the document was AI-written.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from ..data.schema import CLASS4_NAMES

UNCERTAIN = "uncertain"
DEFINITIONS = {
    "class_probabilities": "Overlap-weighted mean of temperature-calibrated per-window four-way probabilities.",
    "ai_signal_score": "Overlap-weighted mean of the calibrated binary AI-involvement probability. A model "
                       "signal, not the probability that anyone cheated.",
    "flagged_text_share": "Share of analysed characters inside windows at or above the deployed threshold. "
                          "Descriptive only; not validated as the proportion of AI-written text.",
}


@dataclass
class Policy:
    threshold: float                 # binary AI-involvement threshold chosen on validation human writing
    operating_point: str             # e.g. "fpr_0.01"
    min_eligible_words: int = 150
    min_class_confidence: float = 0.5


def aggregate(window_p4: np.ndarray, window_pai: np.ndarray, weights: np.ndarray,
              eligible_words: int, policy: Policy) -> dict:
    if eligible_words < policy.min_eligible_words or len(weights) == 0 or weights.sum() <= 0:
        return {"status": "insufficient_evidence", "document_class": UNCERTAIN,
                "class_probabilities": None, "ai_signal_score": None, "flagged_text_share": None,
                "reasons": [f"eligible prose ({eligible_words} words) below the {policy.min_eligible_words}-word minimum"]}
    w = weights / weights.sum()
    p4 = (window_p4 * w[:, None]).sum(axis=0)
    pai = float((window_pai * w).sum())
    flagged = window_pai >= policy.threshold
    share = float(w[flagged].sum())

    reasons = []
    if pai >= policy.threshold:
        ai_classes = p4[1:]
        cls = 1 + int(ai_classes.argmax())
        conf = float(ai_classes.max() / max(ai_classes.sum(), 1e-9))
        if conf < policy.min_class_confidence:
            document_class, reasons = UNCERTAIN, ["AI-involvement signal above threshold but the provenance class is ambiguous"]
        else:
            document_class = CLASS4_NAMES[cls].lower()
    else:
        if int(p4.argmax()) == 0:
            document_class = CLASS4_NAMES[0].lower()
        else:
            document_class = UNCERTAIN
            reasons = ["binary signal below threshold but four-way head does not favour human-only"]
    return {"status": "complete", "document_class": document_class,
            "class_probabilities": {n.lower(): float(v) for n, v in zip(CLASS4_NAMES, p4)},
            "ai_signal_score": pai, "flagged_text_share": share, "reasons": reasons}


### `ml/scriptsense_ml/inference/service.py`
Document-level AI-writing analysis using a trained, calibrated run (CLAUDE.md section 8 contract).

In [ ]:
%%writefile ml/scriptsense_ml/inference/service.py
"""Document-level AI-writing analysis using a trained, calibrated run (CLAUDE.md section 8 contract)."""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer

from ..data.normalization import normalize_for_model, word_count
from ..evaluation.calibration import apply_temperature
from ..models.multitask import MultiTaskDetector
from .aggregation import DEFINITIONS, Policy, aggregate
from .chunking import eligible_spans, offset_to_page, segment_blocks, token_windows
from .predict import inference_amp_dtype, predict_logits

STANDING_LIMITATIONS = [
    "AI-writing analysis is a probabilistic model signal, not proof of authorship or misconduct.",
    "AI-paraphrased/refined and mixed-authorship text are substantially harder to detect than raw AI output.",
    "Validated only on English text from the training/evaluation domains; other genres and languages may be unreliable.",
    "Results are not comparable to, and are not designed to reproduce, any commercial detector's scores.",
]


class ModelNotReadyError(RuntimeError):
    pass


class AIWritingAnalyzer:
    def __init__(self, run_best_dir: Path, device: str | None = None, min_eligible_words: int = 150):
        run_best_dir = Path(run_best_dir)
        cal_path = run_best_dir / "calibration.json"
        if not (run_best_dir / "heads.pt").exists() or not cal_path.exists():
            raise ModelNotReadyError(f"no trained+calibrated model at {run_best_dir}")
        self.meta = json.loads((run_best_dir / "model_meta.json").read_text())
        self.cal = json.loads(cal_path.read_text())
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = MultiTaskDetector.load(run_best_dir).to(self.device).eval()
        self.tokenizer = AutoTokenizer.from_pretrained(run_best_dir / "tokenizer")
        self.max_length = self.meta["config"]["max_length"]
        self.policy = Policy(threshold=self.cal["deployed_threshold"],
                             operating_point=self.cal["deployed_operating_point"],
                             min_eligible_words=min_eligible_words)
        data = self.meta.get("data", {})
        self.model_version = (f"{self.meta['config']['run_name']}"
                              f"+data:{str(data.get('dataset_sha256', ''))[:12]}"
                              f"+cal:{self.cal['fitted_at'][:10]}")
        self.is_subset_model = bool(data.get("is_subset"))

    def analyze(self, text: str, page_starts: list[int] | None = None, document_id: str = "") -> dict:
        page_starts = page_starts or [0]
        blocks = segment_blocks(text)
        spans = eligible_spans(blocks)
        eligible_words = sum(word_count(text[s:e]) for s, e in spans)
        total_words = word_count(text)
        excluded = {}
        for b in blocks:
            if not b.eligible:
                excluded[b.reason] = excluded.get(b.reason, 0) + word_count(text[b.start:b.end])

        windows = [w for span in spans for w in token_windows(text, span, self.tokenizer, self.max_length, 64)]
        limitations = list(STANDING_LIMITATIONS)
        if self.is_subset_model:
            limitations.insert(0, "This model was trained on a documented data SUBSET (smoke/development run); "
                                  "its outputs are not production-validated.")
        base = {
            "document_id": document_id, "model_version": self.model_version,
            "eligible_word_count": eligible_words, "total_word_count": total_words,
            "excluded_word_counts": excluded, "ai_involvement_estimate": None,
            "operating_point": self.policy.operating_point, "threshold": self.policy.threshold,
            "definitions": DEFINITIONS, "limitations": limitations,
        }
        if not windows:
            agg = aggregate(np.zeros((0, 4)), np.zeros(0), np.zeros(0), eligible_words, self.policy)
            return {**base, **agg, "analyzed_word_count": 0, "segments": []}

        chunks = [normalize_for_model(text[w.start:w.end]) for w in windows]
        amp = inference_amp_dtype(self.device)
        l4, l2 = predict_logits(self.model, self.tokenizer, chunks, self.max_length, 16, self.device, amp)
        p4 = apply_temperature(l4, self.cal["temperature_4way"])
        pai = apply_temperature(l2, self.cal["temperature_binary"])[:, 1]
        weights = np.array([w.new_chars for w in windows], dtype=float)
        agg = aggregate(p4, pai, weights, eligible_words, self.policy)

        segments = []
        for w, probs, p in zip(windows, p4, pai):
            flagged = bool(p >= self.policy.threshold)
            segments.append({
                "start": w.start, "end": w.end,
                "page_start": offset_to_page(w.start, page_starts), "page_end": offset_to_page(max(w.end - 1, 0), page_starts),
                "n_tokens": w.n_tokens,
                "class_probabilities": {k.lower(): float(v) for k, v in zip(
                    ["HUMAN_ONLY", "AI_GENERATED", "AI_PARAPHRASED_OR_REFINED", "MIXED_OR_AI_ASSISTED"], probs)},
                "ai_signal": float(p), "flagged": flagged,
                "predicted_class": ["human_only", "ai_generated", "ai_paraphrased_or_refined",
                                    "mixed_or_ai_assisted"][int(probs.argmax())],
                "threshold": self.policy.threshold, "model_version": self.model_version,
            })
        analyzed = word_count(" ".join(text[s:e] for s, e in spans))
        return {**base, **agg, "analyzed_word_count": analyzed, "segments": segments}


## 2. Code: Package: similarity engine (early components)

### `ml/scriptsense_ml/similarity/fingerprints.py`
Word n-gram fingerprints with winnowing (Schleimer, Wilkerson & Aiken, 2003) and exact passage alignment that keeps character offsets into the original texts.

In [ ]:
%%writefile ml/scriptsense_ml/similarity/fingerprints.py
"""Word n-gram fingerprints with winnowing (Schleimer, Wilkerson & Aiken, 2003) and
exact passage alignment that keeps character offsets into the original texts.

Tokens are case/punctuation-normalised words, each remembering its original span, so a
match can always be highlighted in both the submission and the source.
"""
from __future__ import annotations

import re
import unicodedata
import zlib
from dataclasses import dataclass

_WORD = re.compile(r"\w+", re.UNICODE)


@dataclass(frozen=True)
class Token:
    norm: str
    start: int
    end: int


def tokenize(text: str) -> list[Token]:
    return [Token(unicodedata.normalize("NFKC", m.group()).casefold(), m.start(), m.end())
            for m in _WORD.finditer(text)]


def ngram_hashes(tokens: list[Token], n: int) -> list[int]:
    return [zlib.crc32(" ".join(t.norm for t in tokens[i:i + n]).encode()) for i in range(len(tokens) - n + 1)]


def winnow(hashes: list[int], window: int) -> list[tuple[int, int]]:
    """Return (hash, position) fingerprints: the rightmost minimum of each window.
    Guarantees any shared run of >= window + n - 1 tokens shares at least one fingerprint."""
    if not hashes:
        return []
    if len(hashes) <= window:
        m = min(hashes)
        return [(m, max(i for i, h in enumerate(hashes) if h == m))]
    out, last = [], -1
    for s in range(len(hashes) - window + 1):
        win = hashes[s:s + window]
        m = min(win)
        pos = s + max(i for i, h in enumerate(win) if h == m)
        if pos != last:
            out.append((m, pos))
            last = pos
    return out


@dataclass
class Match:
    sub_start: int   # character offsets in the submission
    sub_end: int
    src_start: int   # character offsets in the source
    src_end: int
    n_words: int


def align_exact(sub_tokens: list[Token], src_tokens: list[Token], n: int = 5, min_words: int = 8) -> list[Match]:
    """Maximal runs of identical normalised words (seeded by shared n-grams) of >= min_words."""
    if len(sub_tokens) < n or len(src_tokens) < n:
        return []
    src_index: dict[int, list[int]] = {}
    for j, h in enumerate(ngram_hashes(src_tokens, n)):
        src_index.setdefault(h, []).append(j)
    matches, covered_until = [], -1
    sub_hashes = ngram_hashes(sub_tokens, n)
    i = 0
    while i < len(sub_hashes):
        best = None
        for j in src_index.get(sub_hashes[i], []):
            if any(sub_tokens[i + k].norm != src_tokens[j + k].norm for k in range(n)):
                continue  # crc collision
            # extend left (only within not-yet-covered submission text) and right
            a, b = i, j
            while a > covered_until + 1 and b > 0 and sub_tokens[a - 1].norm == src_tokens[b - 1].norm:
                a, b = a - 1, b - 1
            length = 0
            while a + length < len(sub_tokens) and b + length < len(src_tokens) and \
                    sub_tokens[a + length].norm == src_tokens[b + length].norm:
                length += 1
            if best is None or length > best[2]:
                best = (a, b, length)
        if best and best[2] >= min_words:
            a, b, length = best
            matches.append(Match(sub_tokens[a].start, sub_tokens[a + length - 1].end,
                                 src_tokens[b].start, src_tokens[b + length - 1].end, length))
            covered_until = a + length - 1
            i = a + length
        else:
            i += 1
    return matches


### `ml/scriptsense_ml/similarity/lexical_search.py`
Minimal in-memory BM25 (Okapi) over passages, for lexical candidate retrieval.

In [ ]:
%%writefile ml/scriptsense_ml/similarity/lexical_search.py
"""Minimal in-memory BM25 (Okapi) over passages, for lexical candidate retrieval.

Adequate for a local MVP corpus (thousands to low hundreds of thousands of passages).
Swap for OpenSearch/Elasticsearch when the corpus outgrows memory.
"""
from __future__ import annotations

import math
from collections import Counter, defaultdict

from .fingerprints import tokenize

STOPWORDS = frozenset(
    "a an and are as at be by for from has have in is it its of on or that the this to was were which with".split())


def terms(text: str) -> list[str]:
    return [t.norm for t in tokenize(text) if t.norm not in STOPWORDS]


class BM25Index:
    def __init__(self, k1: float = 1.2, b: float = 0.75):
        self.k1, self.b = k1, b
        self.doc_ids: list[str] = []
        self.doc_len: list[int] = []
        self.postings: dict[str, list[tuple[int, int]]] = defaultdict(list)

    def add(self, doc_id: str, text: str) -> None:
        tf = Counter(terms(text))
        idx = len(self.doc_ids)
        self.doc_ids.append(doc_id)
        self.doc_len.append(sum(tf.values()))
        for term, c in tf.items():
            self.postings[term].append((idx, c))

    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        n = len(self.doc_ids)
        if n == 0:
            return []
        avg = sum(self.doc_len) / n
        scores: dict[int, float] = defaultdict(float)
        for term in set(terms(query)):
            plist = self.postings.get(term)
            if not plist:
                continue
            idf = math.log(1 + (n - len(plist) + 0.5) / (len(plist) + 0.5))
            for d, tf in plist:
                denom = tf + self.k1 * (1 - self.b + self.b * self.doc_len[d] / avg)
                scores[d] += idf * tf * (self.k1 + 1) / denom
        top = sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:k]
        return [(self.doc_ids[d], s) for d, s in top]


## 2. Code: Configs

### `ml/configs/data.yaml`

In [ ]:
%%writefile ml/configs/data.yaml
# ScriptSense AI-detector data budget. Quotas are targets; actual counts are written to
# data/manifests/ and may be lower if a stratum has too few eligible rows. Never pad.
seed: 20261025

raid:
  # Lineage = RAID source_id (the human source document and every generation/attack derived
  # from it). A hash of the lineage decides train-pool vs challenge-pool; no lineage is in both.
  challenge_lineage_fraction: 0.10
  # Validation pool (early stopping, temperature scaling, threshold selection) - lineage-disjoint
  # from both train and challenge so RAID domains are represented in validation.
  validation_lineage_fraction: 0.05
  # Unseen-generator evaluation: these model families never appear in training rows.
  holdout_generators: [cohere, cohere-chat]
  train_quota:
    ai_clean: 50000        # attack == none                       -> AI_GENERATED
    ai_paraphrase: 25000   # attack == paraphrase (DIPPER)        -> AI_PARAPHRASED_OR_REFINED
    ai_other: 25000        # all other attacks, stratified        -> AI_GENERATED or binary-only (synonym)
    # Not in the CLAUDE.md 100k budget; added so RAID domains (recipes, poetry, ...) are not
    # AI-only in training, which would let the model learn "domain => AI". Counted separately.
    # Set to 0 to reproduce the original budget exactly (ablation).
    human_clean: 25000
  validation_quota:
    ai_clean: 1500
    ai_paraphrase: 1000
    ai_other: 1000
    human_clean: 1500
  challenge_quota:          # 20,000 rows total; not used for weight updates or thresholds
    ai_clean: 7000
    ai_paraphrase: 4000
    ai_other: 4000
    human_clean: 5000

txd22:
  # prompt-grouped split of free components
  ratios: {train: 0.8, validation: 0.1, test: 0.1}

quality:
  # Degenerate repetition loops (e.g. GPT-2 greedy "it is a film that is not for everyone, because
  # it is a film ...") are dropped: they are trivially detectable, inflate metrics, and chain
  # unrelated documents into giant near-duplicate components. Ratio = unique word trigrams /
  # trigrams, computed on texts with >= 20 trigrams. Measured on 2026-10-25: < 0.3 affects ~4.1%
  # of sampled RAID rows (almost all AI), 0.15% of SemEval, 0% of TXD-22.
  min_unique_trigram_ratio: 0.3

dedup:
  num_perm: 128
  bands: 16              # 16 bands x 8 rows: LSH candidate threshold ~0.71
  near_dup_threshold: 0.8
  # Near-duplicates are always merged into one split group. Extra copies are dropped only
  # when they also agree on class_4way and label_binary (true duplicates); differing-label
  # near-duplicates (e.g. a clean generation and its homoglyph attack) are kept together.


### `ml/configs/ai_detector.yaml`

In [ ]:
%%writefile ml/configs/ai_detector.yaml
# Starting points from CLAUDE.md section 3 - to be validated, not guarantees.
run_name: deberta_multitask_s1
model_name: microsoft/deberta-v3-base
mode: multitask              # multitask (4-way + binary) | binary (binary head only)
seed: 1
max_length: 384
learning_rate: 2.0e-5
epochs: 3
per_device_batch_size: 4      # measured: 3.48 GB peak on the 4 GB RTX 3050 (bs 8 spills to system RAM)
grad_accum_steps: 4           # effective batch 16
weight_decay: 0.01
warmup_ratio: 0.06
max_grad_norm: 1.0
binary_loss_weight: 0.5
class_weight_power: 0.5      # inverse-sqrt frequency weighting for the 4-way loss
precision: auto              # auto: bf16 on CUDA if supported, else fp16 on CUDA, fp32 on CPU
gradient_checkpointing: true # needed to fit deberta-v3-base @384 on a 4 GB GPU
# Hardware deviation: freezing the 128k x 768 word-embedding matrix (98M of 184M params) saves its
# gradients + AdamW state. Run an unfrozen comparison on larger hardware before final claims.
freeze_word_embeddings: true
early_stopping_patience: 1   # evaluations without improvement
eval_every_steps: null       # null = once per epoch
selection_metric: auto       # auto: 4-way macro F1 (multitask) / binary AUROC (binary)
eval_batch_size: 32
# Resumable training: a checkpoint (~2 GB) is saved every N optimizer steps and at each epoch end;
# re-running the same command with the same run_name continues from it. Deleted after a finished run.
checkpoint_every_steps: 1000
resume: true
keep_last_checkpoint: false
dataloader_workers: 2         # ignored on Windows
max_train_rows: null         # null = all; an integer selects a documented stratified subset
max_val_rows: null
ablation:
  exclude_raid_paraphrase: false
  exclude_txd22_mixed: false
  exclude_sources: []


### `ml/requirements.txt`

In [ ]:
%%writefile ml/requirements.txt
# Training/eval environment (tested 2026-10-25). Install torch from the CUDA index first:
#   pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cu128
huggingface_hub==1.33.0
matplotlib==3.11.2
numpy==2.3.1
pandas==2.3.1
protobuf==7.36.2
pyarrow==21.0.0
pytest==9.1.1
PyYAML==6.0.3
scikit-learn==1.7.1
scipy==1.18.1
sentencepiece==0.2.2
tiktoken==0.14.0
tokenizers==0.22.2
transformers==5.7.0


### `ml/pytest.ini`

In [ ]:
%%writefile ml/pytest.ini
[pytest]
testpaths = tests
pythonpath = .
addopts = -q


## 2. Code: Pipeline scripts

### `ml/scripts/download_datasets.py`
Download raw datasets into SCRIPTSENSE_DATA_DIR/raw and verify published hashes.

In [ ]:
%%writefile ml/scripts/download_datasets.py
"""Download raw datasets into SCRIPTSENSE_DATA_DIR/raw and verify published hashes.

Every source below has a publisher-provided checksum; a download that does not
match is deleted and the script exits non-zero. A provenance record (URL,
license note, size, hash, timestamp) is written to data/manifests/downloads.json.

Usage:
    python ml/scripts/download_datasets.py txd22 semeval raid_train
"""
from __future__ import annotations

import argparse
import hashlib
import json
import sys
import time
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scriptsense_ml.paths import MANIFEST_DIR, RAW_DIR, ensure_dirs  # noqa: E402

SOURCES = {
    "txd22": {
        "url": "https://data.mendeley.com/public-files/datasets/prcjcggtjf/files/"
               "1e842f9d-4ff8-4986-bc7c-cde460f806ee/file_downloaded",
        "dest": "txd22/TXD-22 A Large-Scale Benchmark Dataset.csv",
        "algo": "sha256",
        "hash": "232c260f8caf7ea61a331f39fa730956862afbe0c5fc554720daa07d215a2f7d",
        "size": 121656267,
        "landing": "https://data.mendeley.com/datasets/prcjcggtjf/1",
        "license": "CC BY 4.0 (per Mendeley Data record; attribution required)",
        "version": "Mendeley Data v1, DOI 10.17632/prcjcggtjf.1, published 2026-07-01",
    },
    "semeval": {
        "url": "https://zenodo.org/records/11002158/files/SemEval2024-Task8-data.zip?download=1",
        "dest": "semeval2024_task8/SemEval2024-Task8-data.zip",
        "algo": "md5",
        "hash": "a126d3369b55931ac43da585aededef6",
        "size": 419460633,
        "landing": "https://zenodo.org/records/11002158",
        "license": "UNSTATED in Zenodo record metadata; user confirmed research use on 2026-10-25. "
                   "Do not redistribute until terms are confirmed.",
        "version": "Zenodo record 11002158",
    },
    "raid_train": {
        "url": "https://huggingface.co/datasets/liamdugan/raid/resolve/main/train.csv",
        "dest": "raid/train.csv",
        "algo": "sha256",
        "hash": "52f04ceebc126064e68fbd22d8b736964065745464f4bfd52e488150b49f84e4",
        "size": 11779491051,
        "landing": "https://huggingface.co/datasets/liamdugan/raid",
        "license": "MIT (per Hugging Face dataset card); generations derive from third-party "
                   "source texts - confirm terms before redistribution",
        "version": "HF liamdugan/raid main, LFS oid above",
    },
}


def _hash_file(path: Path, algo: str) -> str:
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def download(name: str) -> dict:
    spec = SOURCES[name]
    dest = RAW_DIR / spec["dest"]
    dest.parent.mkdir(parents=True, exist_ok=True)

    if dest.exists() and dest.stat().st_size == spec["size"]:
        print(f"[{name}] present, verifying {spec['algo']}...", flush=True)
    else:
        part = dest.with_suffix(dest.suffix + ".part")
        start = part.stat().st_size if part.exists() else 0
        req = urllib.request.Request(spec["url"], headers={"User-Agent": "scriptsense-data/1.0"})
        if start:
            req.add_header("Range", f"bytes={start}-")
        print(f"[{name}] downloading {spec['size'] / 1e6:.1f} MB (resume at {start})", flush=True)
        with urllib.request.urlopen(req, timeout=60) as resp:
            if start and resp.status != 206:
                start = 0  # server ignored Range; restart
            with open(part, "ab" if start else "wb") as out:
                done, last = start, time.time()
                while block := resp.read(4 * 1024 * 1024):
                    out.write(block)
                    done += len(block)
                    if time.time() - last > 30:
                        print(f"[{name}] {done / spec['size']:.1%}", flush=True)
                        last = time.time()
        part.replace(dest)

    digest = _hash_file(dest, spec["algo"])
    if digest != spec["hash"]:
        dest.unlink()
        raise SystemExit(f"[{name}] HASH MISMATCH ({digest} != {spec['hash']}); file deleted")
    print(f"[{name}] OK {dest}", flush=True)
    return {
        "name": name,
        "url": spec["url"],
        "landing_page": spec["landing"],
        "version": spec["version"],
        "license": spec["license"],
        "local_path": str(dest.relative_to(RAW_DIR)),
        "size_bytes": dest.stat().st_size,
        "hash_algo": spec["algo"],
        "hash": digest,
        "verified_at": datetime.now(timezone.utc).isoformat(),
    }


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("names", nargs="+", choices=sorted(SOURCES))
    args = ap.parse_args()
    ensure_dirs()
    manifest_path = MANIFEST_DIR / "downloads.json"
    for name in args.names:
        record = download(name)
        # Re-read right before writing so concurrent downloads don't clobber each other.
        records = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
        records[name] = record
        manifest_path.write_text(json.dumps(records, indent=2, sort_keys=True))


if __name__ == "__main__":
    main()


### `ml/scripts/inspect_datasets.py`
Profile raw datasets before any training: schema, counts, labels, lengths, missing values, model/domain/attack distribution and license. Writes data/manifests/raw_profile_<source>.json.

In [ ]:
%%writefile ml/scripts/inspect_datasets.py
"""Profile raw datasets before any training: schema, counts, labels, lengths, missing values,
model/domain/attack distribution and license. Writes data/manifests/raw_profile_<source>.json.

Usage:
    python ml/scripts/inspect_datasets.py txd22 semeval raid
"""
from __future__ import annotations

import argparse
import json
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scriptsense_ml.data import sources  # noqa: E402
from scriptsense_ml.data.schema import CLASS4_NAMES, UNKNOWN  # noqa: E402
from scriptsense_ml.paths import MANIFEST_DIR, RAW_DIR  # noqa: E402

TXD22_PATH = RAW_DIR / "txd22" / "TXD-22 A Large-Scale Benchmark Dataset.csv"
SEMEVAL_ZIP = RAW_DIR / "semeval2024_task8" / "SemEval2024-Task8-data.zip"
RAID_TRAIN = RAW_DIR / "raid" / "train.csv"


def _length_stats(words: pd.Series) -> dict:
    q = words.quantile([0, 0.05, 0.25, 0.5, 0.75, 0.95, 1]).round(1)
    return {"mean": round(float(words.mean()), 1), **{f"p{int(k * 100)}": float(v) for k, v in q.items()}}


def _class_name(c: int) -> str:
    return "UNKNOWN" if c == UNKNOWN else CLASS4_NAMES[c]


def profile_rows(rows: list[dict], extra: dict) -> dict:
    df = pd.DataFrame(rows)
    df["words"] = df["text"].str.split().str.len()
    df["class_name"] = df["class_4way"].map(_class_name)
    return {
        **extra,
        "rows": len(df),
        "columns": list(df.columns),
        "empty_text_rows": int((df["text"].str.strip() == "").sum()),
        "rows_by_split": df["split"].replace("", "unassigned").value_counts().to_dict(),
        "source_label_counts": df["source_label"].value_counts().to_dict(),
        "class_4way_counts": df["class_name"].value_counts().to_dict(),
        "binary_counts": df["label_binary"].map({0: "HUMAN_ONLY", 1: "AI_INVOLVED", UNKNOWN: "UNKNOWN"})
                          .value_counts().to_dict(),
        "class_by_split": pd.crosstab(df["split"].replace("", "unassigned"), df["class_name"]).to_dict(),
        "generator_counts": df["generator"].value_counts().to_dict(),
        "domain_counts": df["domain"].replace("", "unknown").value_counts().to_dict(),
        "attack_or_edit_counts": df["attack_or_edit_type"].value_counts().to_dict(),
        "word_length_overall": _length_stats(df["words"]),
        "median_words_by_class": df.groupby("class_name")["words"].median().to_dict(),
        "exact_duplicate_texts_within_source": int(df["text"].duplicated().sum()),
        "license": rows[0]["license_reference"] if rows else "",
    }


def inspect_txd22() -> dict:
    rows, meta = sources.read_txd22(TXD22_PATH)
    df = pd.DataFrame(rows)
    per_prompt = df.groupby("prompt_id").size()
    meta.update({
        "path": str(TXD22_PATH.relative_to(RAW_DIR)),
        "unique_prompts": int(df["prompt_id"].nunique()),
        "answers_per_prompt": per_prompt.describe().round(2).to_dict(),
    })
    return profile_rows(rows, meta)


def inspect_semeval() -> dict:
    with __import__("zipfile").ZipFile(SEMEVAL_ZIP) as zf:
        members = [m for m in zf.namelist() if not m.endswith("/")]
    rows = [sources.semeval_a_row(split, rec) for split, rec in sources.iter_semeval_a(SEMEVAL_ZIP)]
    if not rows:
        raise SystemExit(f"No Subtask A monolingual files found. Zip members: {members[:50]}")
    return profile_rows(rows, {"path": str(SEMEVAL_ZIP.relative_to(RAW_DIR)), "zip_members": members})


RAID_META_COLS = ["id", "adv_source_id", "source_id", "model", "decoding", "repetition_penalty",
                  "attack", "domain"]


def inspect_raid(chunksize: int = 200_000) -> dict:
    """Streaming profile; the text column is read only to measure length."""
    counts = {k: Counter() for k in ("model", "attack", "domain", "decoding", "repetition_penalty")}
    model_attack, missing = Counter(), Counter()
    word_samples, ids, n = [], set(), 0
    dup_ids = 0
    lineages = set()
    rng = np.random.default_rng(0)
    for chunk in pd.read_csv(RAID_TRAIN, chunksize=chunksize, dtype=str, keep_default_na=False,
                             usecols=RAID_META_COLS + ["generation"]):
        n += len(chunk)
        for k in counts:
            counts[k].update(chunk[k].replace("", "<empty>"))
        model_attack.update(zip(chunk["model"], chunk["attack"]))
        for c in chunk.columns:
            missing[c] += int((chunk[c] == "").sum())
        for i in chunk["id"]:
            if i in ids:
                dup_ids += 1
            ids.add(i)
        lineages.update(chunk["source_id"])
        take = rng.random(len(chunk)) < 0.01
        word_samples.extend(chunk.loc[take, "generation"].str.split().str.len().tolist())
        print(f"  raid profiled {n:,} rows", flush=True)
    return {
        "path": str(RAID_TRAIN.relative_to(RAW_DIR)),
        "rows": n,
        "unique_ids": len(ids),
        "duplicate_ids": dup_ids,
        "unique_source_ids (lineages)": len(lineages),
        "empty_values_by_column": dict(missing),
        **{f"{k}_counts": dict(v) for k, v in counts.items()},
        "model_x_attack_counts": {f"{m}|{a}": c for (m, a), c in sorted(model_attack.items())},
        "word_length_1pct_sample": _length_stats(pd.Series(word_samples)),
        "license": sources.RAID_LICENSE,
    }


INSPECTORS = {"txd22": inspect_txd22, "semeval": inspect_semeval, "raid": inspect_raid}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("names", nargs="+", choices=sorted(INSPECTORS))
    args = ap.parse_args()
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    for name in args.names:
        print(f"== {name}", flush=True)
        prof = INSPECTORS[name]()
        out = MANIFEST_DIR / f"raw_profile_{name}.json"
        out.write_text(json.dumps(prof, indent=2, default=str))
        summary = {k: prof[k] for k in prof if k in ("rows", "class_4way_counts", "binary_counts",
                                                      "rows_by_split", "unique_prompts", "decode_line_stats",
                                                      "duplicate_ids", "attack_counts", "model_counts")}
        print(json.dumps(summary, indent=2, default=str))
        print(f"wrote {out}")


if __name__ == "__main__":
    main()


### `ml/scripts/prepare_semeval.py`
SemEval-2024 Task 8 Subtask A monolingual -> canonical rows (processed/semeval_a.parquet).

In [ ]:
%%writefile ml/scripts/prepare_semeval.py
"""SemEval-2024 Task 8 Subtask A monolingual -> canonical rows (processed/semeval_a.parquet).

Publisher splits are kept and pinned: train -> train, dev -> validation, gold test -> test.
"""
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scriptsense_ml.data.io import PREPROCESSING_VERSION, rows_to_frame, write_table  # noqa: E402
from scriptsense_ml.data.sources import iter_semeval_a, semeval_a_row  # noqa: E402
from scriptsense_ml.paths import MANIFEST_DIR, PROCESSED_DIR, RAW_DIR  # noqa: E402

if __name__ == "__main__":
    zip_path = RAW_DIR / "semeval2024_task8" / "SemEval2024-Task8-data.zip"
    rows, empty = [], 0
    for split, rec in iter_semeval_a(zip_path):
        if not str(rec.get("text", "")).strip():
            empty += 1
            continue
        rows.append(semeval_a_row(split, rec))
    df = rows_to_frame(rows)
    out = PROCESSED_DIR / "semeval_a.parquet"
    sha = write_table(df, out)
    record = {"source": "semeval_a", "rows": len(df), "dropped_empty_text": empty,
              "rows_by_split": dict(Counter(df["split"])), "output": out.name, "sha256": sha,
              "preprocessing_version": PREPROCESSING_VERSION}
    (MANIFEST_DIR / "prepared_semeval_a.json").write_text(json.dumps(record, indent=2))
    print(json.dumps(record, indent=2))


### `ml/scripts/prepare_txd22.py`
TXD-22 -> canonical rows (processed/txd22.parquet). Splits are assigned later, grouped by prompt.

In [ ]:
%%writefile ml/scripts/prepare_txd22.py
"""TXD-22 -> canonical rows (processed/txd22.parquet). Splits are assigned later, grouped by prompt."""
from __future__ import annotations

import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scriptsense_ml.data.io import PREPROCESSING_VERSION, rows_to_frame, write_table  # noqa: E402
from scriptsense_ml.data.sources import read_txd22  # noqa: E402
from scriptsense_ml.paths import MANIFEST_DIR, PROCESSED_DIR, RAW_DIR  # noqa: E402

if __name__ == "__main__":
    rows, meta = read_txd22(RAW_DIR / "txd22" / "TXD-22 A Large-Scale Benchmark Dataset.csv")
    df = rows_to_frame(rows)
    out = PROCESSED_DIR / "txd22.parquet"
    sha = write_table(df, out)
    record = {"source": "txd22", "rows": len(df), "output": out.name, "sha256": sha,
              "preprocessing_version": PREPROCESSING_VERSION, **meta}
    (MANIFEST_DIR / "prepared_txd22.json").write_text(json.dumps(record, indent=2))
    print(json.dumps(record, indent=2))


### `ml/scripts/prepare_raid.py`
Stratified, lineage-safe RAID sample -> processed/raid.parquet.

In [ ]:
%%writefile ml/scripts/prepare_raid.py
"""Stratified, lineage-safe RAID sample -> processed/raid.parquet.

Pass 1 streams only metadata columns and counts eligible rows per stratum.
Quotas are spread across strata by water-filling (equal share, leftovers
redistributed to strata that still have rows). Pass 2 streams again and keeps,
per stratum, the rows with the smallest seeded SHA-256 keys - deterministic and
independent of file order. Rows of a challenge lineage or a held-out generator
family can never enter the training pool.
"""
from __future__ import annotations

import hashlib
import heapq
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scriptsense_ml.data import label_mapping as lm  # noqa: E402
from scriptsense_ml.data.normalization import normalize_for_model  # noqa: E402
from scriptsense_ml.data.io import PREPROCESSING_VERSION, rows_to_frame, write_table  # noqa: E402
from scriptsense_ml.data.sources import raid_row  # noqa: E402
from scriptsense_ml.paths import MANIFEST_DIR, ML_ROOT, PROCESSED_DIR, RAW_DIR  # noqa: E402

RAID_TRAIN = RAW_DIR / "raid" / "train.csv"
META = ["id", "source_id", "model", "attack", "domain"]
CHUNK = 200_000


def _u(seed: int, key: str) -> float:
    return int.from_bytes(hashlib.sha256(f"{seed}:{key}".encode()).digest()[:8], "big") / 2**64


def category(model: str, attack: str) -> str | None:
    human = model == "human"
    if attack == "none":
        return "human_clean" if human else "ai_clean"
    if human:
        return None  # human attacked rows are outside this budget
    if attack == "paraphrase":
        return "ai_paraphrase"
    return "ai_other"


def stratum(cat: str, model: str, attack: str, domain: str) -> str:
    if cat == "ai_other":
        return f"{model}|{attack}|{domain}"
    if cat == "human_clean":
        return domain
    return f"{model}|{domain}"


def classify(rec, cfg) -> tuple[str, str, str] | None:
    """Return (pool, category, stratum) or None if the row is ineligible."""
    cat = category(rec.model, rec.attack)
    if cat is None:
        return None
    u = _u(cfg["seed"], "lineage:" + rec.source_id)
    c_frac = cfg["raid"]["challenge_lineage_fraction"]
    holdout = rec.model in cfg["raid"]["holdout_generators"]
    if u < c_frac:
        pool = "challenge"
    elif holdout:
        return None  # held-out generator outside challenge lineages: usable by no pool
    elif u < c_frac + cfg["raid"]["validation_lineage_fraction"]:
        pool = "validation"
    else:
        pool = "train"
    return pool, cat, stratum(cat, rec.model, rec.attack, rec.domain)


def water_fill(available: dict[str, int], quota: int) -> dict[str, int]:
    alloc = {s: 0 for s in available}
    remaining, open_ = quota, {s for s, n in available.items() if n > 0}
    while remaining > 0 and open_:
        share = max(1, remaining // len(open_))
        for s in sorted(open_):
            take = min(share, available[s] - alloc[s], remaining)
            alloc[s] += take
            remaining -= take
            if alloc[s] >= available[s]:
                open_.discard(s)
            if remaining == 0:
                break
    return alloc


def main() -> None:
    cfg = yaml.safe_load((ML_ROOT / "configs" / "data.yaml").read_text())
    quotas = {"train": cfg["raid"]["train_quota"], "validation": cfg["raid"]["validation_quota"],
              "challenge": cfg["raid"]["challenge_quota"]}

    # ---- pass 1: counts
    counts: dict[tuple[str, str], Counter] = defaultdict(Counter)
    total = 0
    for chunk in pd.read_csv(RAID_TRAIN, usecols=META, dtype=str, keep_default_na=False, chunksize=CHUNK):
        for rec in chunk.itertuples(index=False):
            c = classify(rec, cfg)
            if c:
                counts[(c[0], c[1])][c[2]] += 1
        total += len(chunk)
        print(f"pass1 {total:,}", flush=True)

    alloc: dict[tuple[str, str], dict[str, int]] = {}
    for pool, q in quotas.items():
        for cat, n in q.items():
            alloc[(pool, cat)] = water_fill(dict(counts[(pool, cat)]), int(n))

    # ---- pass 2: keep smallest keys per stratum (max-heap of negated keys)
    heaps: dict[tuple[str, str, str], list] = defaultdict(list)
    empty_after_norm: Counter = Counter()
    seen = 0
    for chunk in pd.read_csv(RAID_TRAIN, dtype=str, keep_default_na=False, chunksize=CHUNK):
        for line_no, rec in zip(range(seen, seen + len(chunk)), chunk.itertuples(index=False)):
            c = classify(rec, cfg)
            if not c:
                continue
            pool, cat, st = c
            cap = alloc[(pool, cat)].get(st, 0)
            if cap == 0:
                continue
            key = _u(cfg["seed"], f"row:{rec.id}:{line_no}")
            h = heaps[(pool, cat, st)]
            if len(h) >= cap and -h[0][0] <= key:
                continue
            if not normalize_for_model(rec.generation):
                empty_after_norm[(pool, cat)] += 1  # e.g. only invisible characters
                continue
            item = (-key, line_no, rec._asdict())
            if len(h) < cap:
                heapq.heappush(h, item)
            elif -h[0][0] > key:
                heapq.heapreplace(h, item)
        seen += len(chunk)
        print(f"pass2 {seen:,}", flush=True)

    rows, skipped_empty, selected, used_ids = [], 0, Counter(), set()
    for (pool, cat, st), h in heaps.items():
        for _, line_no, rec in h:
            if not normalize_for_model(rec["generation"]):
                skipped_empty += 1
                continue
            row = raid_row(rec, line_no)
            if row is None:
                continue
            if row["row_id"] in used_ids:  # RAID ids are expected unique; disambiguate if not
                row["source_row_id"] = f"{rec['id']}@{line_no}"
                row["row_id"] = f"raid:{row['source_row_id']}"
            used_ids.add(row["row_id"])
            selected[(pool, cat)] += 1
            row["split"] = pool
            row["split_fixed"] = True
            row["attack_or_edit_type"] = rec["attack"]
            rows.append(row)
    rows.sort(key=lambda r: r["row_id"])
    df = rows_to_frame(rows)
    out = PROCESSED_DIR / "raid.parquet"
    sha = write_table(df, out)

    report = {
        "source": "raid", "raw_rows_scanned": total, "rows": len(df), "output": out.name, "sha256": sha,
        "preprocessing_version": PREPROCESSING_VERSION, "config": cfg["raid"], "seed": cfg["seed"],
        "skipped_empty_text": skipped_empty,
        "skipped_empty_after_normalization_during_selection": {f"{p}/{c}": n for (p, c), n in empty_after_norm.items()},
        "target_vs_actual": {
            f"{pool}/{cat}": {"target": int(n), "eligible": int(sum(counts[(pool, cat)].values())),
                              "selected": selected[(pool, cat)]}
            for pool, q in quotas.items() for cat, n in q.items()
        },
        "class_4way_by_split": pd.crosstab(df["split"], df["class_4way"]).to_dict(),
        "generator_by_split": pd.crosstab(df["generator"], df["split"]).to_dict(),
        "attack_by_split": pd.crosstab(df["attack_or_edit_type"], df["split"]).to_dict(),
        "domain_by_split": pd.crosstab(df["domain"], df["split"]).to_dict(),
        "strata_fully_used": {
            f"{p}/{c}": {s: n for s, n in counts[(p, c)].items() if alloc[(p, c)].get(s, 0) >= n}
            for (p, c) in alloc
        },
    }
    lineage_pools = df.groupby("original_document_id")["split"].nunique()
    report["lineages_in_both_pools"] = int((lineage_pools > 1).sum())
    if report["lineages_in_both_pools"]:
        raise SystemExit("RAID lineage leakage between pools")
    (MANIFEST_DIR / "prepared_raid.json").write_text(json.dumps(report, indent=2, default=str))
    print(json.dumps({k: report[k] for k in ("rows", "target_vs_actual", "lineages_in_both_pools")}, indent=2))


if __name__ == "__main__":
    main()


### `ml/scripts/deduplicate_and_split.py`
Merge prepared sources, deduplicate, assign lineage-grouped splits, audit leakage.

In [ ]:
%%writefile ml/scripts/deduplicate_and_split.py
"""Merge prepared sources, deduplicate, assign lineage-grouped splits, audit leakage.

Policy (also recorded in the report):
  1. Exact duplicates of the *model text* (normalize_for_model output) are collapsed to one
     row when labels agree; when labels conflict every copy is dropped as label noise.
  2. Rows with the same aggressive dedup key (case/punctuation-insensitive) or a MinHash
     near-duplicate (estimated Jaccard >= threshold) are always placed in one split group.
  3. Cross-source near-duplicates with agreeing labels are collapsed to one row; with
     conflicting labels all copies are dropped. Within-source near-duplicates are kept
     (e.g. a RAID generation and its perturbed variants are intentionally distinct samples).
  4. Components inherit publisher-fixed splits; an evaluation pin beats a train pin, so
     official validation/test material is never trained on. Components pinned to two
     different evaluation splits are dropped and reported.
  5. The script fails if any group key spans more than one split.

Outputs: processed/dataset.parquet, data/manifests/dataset_report.json, data/manifests/splits.csv.gz
"""
from __future__ import annotations

import json
import os
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scriptsense_ml.data.deduplication import MinHasher, UnionFind, near_duplicate_pairs  # noqa: E402
from scriptsense_ml.data.io import PREPROCESSING_VERSION, file_sha256, write_table  # noqa: E402
from scriptsense_ml.data.normalization import text_hash, unique_trigram_ratio  # noqa: E402
from scriptsense_ml.data.schema import CLASS4_NAMES, UNKNOWN  # noqa: E402
from scriptsense_ml.data.splitting import assert_no_leakage, assign_splits, connected_components, leakage_report  # noqa: E402
from scriptsense_ml.paths import MANIFEST_DIR, ML_ROOT, PROCESSED_DIR  # noqa: E402

SOURCES = ["semeval_a", "raid", "txd22", "academic"]
_HASHER: MinHasher | None = None


def _init(num_perm: int, seed: int) -> None:
    global _HASHER
    _HASHER = MinHasher(num_perm=num_perm, seed=seed)


def _sigs(texts: list[str]) -> np.ndarray:
    return np.stack([_HASHER.signature(t) for t in texts])


def minhash_all(texts: list[str], num_perm: int, seed: int, workers: int | None = None) -> np.ndarray:
    workers = workers or max(1, min(12, os.cpu_count() or 2))  # Colab has only 2 CPUs
    batches = [texts[i:i + 2000] for i in range(0, len(texts), 2000)]
    with ProcessPoolExecutor(max_workers=workers, initializer=_init, initargs=(num_perm, seed)) as ex:
        return np.concatenate(list(ex.map(_sigs, batches)))


def label_key(df: pd.DataFrame) -> pd.Series:
    return df["class_4way"].astype(str) + "/" + df["label_binary"].astype(str)


def distribution(df: pd.DataFrame) -> dict:
    cname = df["class_4way"].map(lambda c: "UNKNOWN" if c == UNKNOWN else CLASS4_NAMES[c])
    return {
        "rows_by_split": df["split"].value_counts().to_dict(),
        "rows_by_source_split": {f"{s}/{p}": int(n) for (s, p), n in df.groupby(["source_dataset", "split"]).size().items()},
        "class_4way_by_split": pd.crosstab(cname, df["split"]).to_dict(),
        "binary_by_split": pd.crosstab(df["label_binary"], df["split"]).to_dict(),
        "class_4way_by_source_split": {f"{s}/{p}": g.value_counts().to_dict()
                                       for (s, p), g in cname.groupby([df["source_dataset"], df["split"]])},
        "domain_by_split": pd.crosstab(df["domain"].replace("", "unknown"), df["split"]).to_dict(),
        "generator_by_split": pd.crosstab(df["generator"].replace("", "unknown"), df["split"]).to_dict(),
        "attack_by_split": pd.crosstab(df["attack_or_edit_type"], df["split"]).to_dict(),
        "median_words_by_class_split": {f"{c}/{s}": float(v) for (c, s), v in
                                        df["text"].str.split().str.len().groupby([cname, df["split"]]).median().items()},
    }


def main() -> None:
    t0 = time.time()
    cfg = yaml.safe_load((ML_ROOT / "configs" / "data.yaml").read_text())
    dcfg, seed = cfg["dedup"], cfg["seed"]

    frames, inputs = [], {}
    for name in SOURCES:
        p = PROCESSED_DIR / f"{name}.parquet"
        if p.exists():
            frames.append(pd.read_parquet(p))
            inputs[name] = {"file": p.name, "sha256": file_sha256(p), "rows": len(frames[-1])}
    missing = [s for s in SOURCES if s not in inputs]
    df = pd.concat(frames, ignore_index=True)
    report: dict = {"created_at": datetime.now(timezone.utc).isoformat(), "seed": seed,
                    "preprocessing_version": PREPROCESSING_VERSION, "inputs": inputs,
                    "sources_missing": missing, "rows_before": len(df),
                    "rows_before_by_source_split": {f"{s}/{p or 'unassigned'}": int(n) for (s, p), n in
                                                    df.groupby(["source_dataset", "split"]).size().items()}}
    print(f"loaded {len(df):,} rows from {list(inputs)}", flush=True)
    dropped = Counter()

    # ---- 0. quality filter: degenerate repetition loops
    rep = df["text"].map(unique_trigram_ratio)
    degenerate = rep < cfg["quality"]["min_unique_trigram_ratio"]
    dropped["degenerate_repetition"] = int(degenerate.sum())
    report["degenerate_repetition_by_source_class"] = {
        f"{s}/{c}": int(n) for (s, c), n in df[degenerate].groupby(["source_dataset", "class_4way"]).size().items()}
    df = df[~degenerate].reset_index(drop=True)

    # ---- 1. exact duplicates of model text
    df["model_hash"] = df["text"].map(lambda t: __import__("hashlib").sha1(t.encode()).hexdigest())
    lk = label_key(df)
    conflict_hashes = set(df.assign(lk=lk).groupby("model_hash")["lk"].nunique().loc[lambda s: s > 1].index)
    is_conflict = df["model_hash"].isin(conflict_hashes)
    dropped["exact_dup_conflicting_labels"] = int(is_conflict.sum())
    df = df[~is_conflict]
    # Prefer keeping evaluation-pinned copies so the split pin survives the collapse.
    pri = df["split"].map({"test": 0, "challenge": 1, "validation": 2}).fillna(3)
    df = df.assign(_pri=pri).sort_values(["_pri", "row_id"])
    dup_mask = df.duplicated("model_hash", keep="first")
    # Keep the dropped copies' group keys on the survivor so their lineage still constrains splits.
    extra_keys = df[dup_mask].groupby("model_hash")["group_keys"].agg("|".join)
    dropped["exact_dup_same_labels"] = int(dup_mask.sum())
    df = df[~dup_mask].drop(columns="_pri").reset_index(drop=True)
    df["group_keys"] = [g + ("|" + extra_keys[h] if h in extra_keys.index else "")
                        for g, h in zip(df["group_keys"], df["model_hash"])]
    print(f"exact dedup: {dict(dropped)} -> {len(df):,}", flush=True)

    # ---- 2. aggressive-key groups + MinHash near-duplicates
    df["dedup_hash"] = df["text"].map(text_hash)
    sigs = minhash_all(df["text"].tolist(), dcfg["num_perm"], seed)
    print(f"minhash done {time.time() - t0:.0f}s", flush=True)
    pairs = near_duplicate_pairs(sigs, bands=dcfg["bands"], threshold=dcfg["near_dup_threshold"])
    del sigs
    src = df["source_dataset"].to_numpy()
    labels = label_key(df).to_numpy()
    uf = UnionFind(len(df))
    cross_conflict, cross_same, pair_stats = set(), [], Counter()
    for i, j, _ in pairs:
        uf.union(i, j)
        cross = src[i] != src[j]
        same = labels[i] == labels[j]
        pair_stats[f"{'cross' if cross else 'within'}_source/{'same' if same else 'diff'}_labels"] += 1
        if cross and not same:
            cross_conflict.update((i, j))
        elif cross and same:
            cross_same.append((i, j))
    report["near_duplicate_pairs"] = dict(pair_stats)
    near_root = [uf.find(i) for i in range(len(df))]
    df["group_keys"] = [f"{g}|dup:exact:{h}|dup:near:{df.at[r, 'row_id']}" for g, h, r in
                        zip(df["group_keys"], df["dedup_hash"], near_root)]

    drop_idx = set(cross_conflict)
    dropped["cross_source_near_dup_conflicting_labels"] = len(cross_conflict)
    # collapse cross-source same-label near-dups: keep the higher-priority copy of each pair
    pri = df["split"].map({"test": 0, "challenge": 1, "validation": 2}).fillna(3).to_numpy()
    n_same = 0
    for i, j in cross_same:
        if i in drop_idx or j in drop_idx:
            continue
        loser = j if (pri[i], df.at[i, "row_id"]) <= (pri[j], df.at[j, "row_id"]) else i
        drop_idx.add(loser)
        n_same += 1
    dropped["cross_source_near_dup_same_labels"] = n_same
    keep = ~df.index.isin(sorted(drop_idx))
    df = df[keep].reset_index(drop=True)
    print(f"near-dup: pairs={dict(pair_stats)} dropped={dict(dropped)} -> {len(df):,}", flush=True)

    # ---- 3. components and split assignment
    keys = [k.split("|") for k in df["group_keys"]]
    comps = connected_components(keys)
    row_ids = df["row_id"].tolist()
    comp_name: dict[int, str] = {}
    for c, rid in zip(comps, row_ids):
        comp_name[c] = min(comp_name.get(c, rid), rid)
    fixed = [s if f else "" for s, f in zip(df["split"], df["split_fixed"])]
    splits, conflicts = assign_splits(comps, comp_name, fixed, cfg["txd22"]["ratios"], seed)
    moved = sum(1 for f, s in zip(fixed, splits) if f and s and f != s)
    audit = df.assign(new_split=splits, component=comps, fixed_split=fixed)
    audit = audit[(audit.new_split == "") | ((audit.fixed_split != "") & (audit.fixed_split != audit.new_split))]
    comp_sizes = Counter(comps)
    audit = audit.assign(component_size=audit.component.map(comp_sizes),
                         words=audit.text.str.split().str.len(), text_head=audit.text.str[:120])
    audit[["row_id", "source_dataset", "generator", "domain", "attack_or_edit_type", "class_4way", "fixed_split",
           "new_split", "component", "component_size", "words", "text_head"]].to_csv(
        MANIFEST_DIR / "split_moves_and_conflicts.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})
    report["largest_components"] = [{"size": n, "splits_pinned": sorted(set(f for f, c in zip(fixed, comps) if c == comp and f))}
                                    for comp, n in comp_sizes.most_common(5)]
    df["split"] = splits
    dropped["component_conflicting_eval_pins"] = int((df["split"] == "").sum())
    df = df[df["split"] != ""].reset_index(drop=True)
    report["pinned_rows_moved_to_eval_split"] = moved
    report["conflicting_components"] = len(conflicts)
    report["components"] = len(set(comps))

    # ---- 4. leakage audit (hard failure)
    keys = [k.split("|") for k in df["group_keys"]]
    report["leakage"] = leakage_report(df["split"].tolist(), keys)
    assert_no_leakage(df["split"].tolist(), keys)
    for col_a, col_b in (("dedup_hash", "split"), ("model_hash", "split")):
        n = int((df.groupby(col_a)[col_b].nunique() > 1).sum())
        report["leakage"][f"{col_a}_spanning_splits"] = n
        assert n == 0, f"{col_a} spans splits"

    # ---- 5. outputs
    report["dropped"] = dict(dropped)
    report["rows_after"] = len(df)
    report["distribution_after"] = distribution(df)
    report["train_rows_after_dedup"] = int((df["split"] == "train").sum())
    report["train_rows_target_before_dedup"] = 284557
    report["notes"] = [
        "Counts are actual post-filter counts; the 284,557 figure is the planning target only.",
        "academic source is absent unless processed/academic.parquet exists (requires consented human data).",
        "RAID human_clean training rows are an addition to the CLAUDE.md budget (domain-shortcut mitigation); see ml/configs/data.yaml.",
    ]
    out = PROCESSED_DIR / "dataset.parquet"
    report["output"] = {"file": out.name, "sha256": write_table(df.drop(columns=["model_hash"]), out)}
    df[["row_id", "source_dataset", "split", "class_4way", "label_binary"]].to_csv(
        MANIFEST_DIR / "splits.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})
    report["split_manifest"] = {"file": "splits.csv.gz", "sha256": file_sha256(MANIFEST_DIR / "splits.csv.gz")}
    report["elapsed_seconds"] = round(time.time() - t0, 1)
    (MANIFEST_DIR / "dataset_report.json").write_text(json.dumps(report, indent=2, default=str))
    print(json.dumps({k: report[k] for k in ("rows_before", "dropped", "rows_after", "train_rows_after_dedup",
                                             "pinned_rows_moved_to_eval_split", "conflicting_components")}, indent=2))
    print(json.dumps(report["distribution_after"]["rows_by_source_split"], indent=2))


if __name__ == "__main__":
    main()


### `ml/scripts/train_detector.py`
Train the AI-writing detector on processed/dataset.parquet.

In [ ]:
%%writefile ml/scripts/train_detector.py
"""Train the AI-writing detector on processed/dataset.parquet.

Usage:
    python ml/scripts/train_detector.py --config ml/configs/ai_detector.yaml [--set key=value ...]
e.g. --set model_name=roberta-base --set mode=binary --set run_name=roberta_binary_s1
     --set max_train_rows=2000 --set max_val_rows=500   (documented smoke subset)

Refuses to start if any normalized text appears in both train and an evaluation split.
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scriptsense_ml.data.io import file_sha256  # noqa: E402
from scriptsense_ml.data.normalization import text_hash  # noqa: E402
from scriptsense_ml.paths import ARTIFACT_DIR, MANIFEST_DIR, PROCESSED_DIR  # noqa: E402
from scriptsense_ml.training.data import (  # noqa: E402
    apply_ablation, load_dataset, row_ids_digest, stratified_subset,
)
from scriptsense_ml.training.trainer import train  # noqa: E402


def parse_overrides(pairs: list[str]) -> dict:
    out = {}
    for p in pairs:
        k, v = p.split("=", 1)
        out[k] = yaml.safe_load(v)
    return out


def check_text_leakage(df, ds_path: Path, ds_sha: str) -> None:
    """Fail if any normalized text is in train and an evaluation split. The (slow) result is cached
    next to the dataset, keyed by its SHA-256, so a changed dataset is always re-checked."""
    cache = ds_path.with_name(ds_path.name + ".leakage_ok.json")
    if cache.exists() and json.loads(cache.read_text()).get("sha256") == ds_sha:
        print("leakage check: cached pass for this exact dataset", flush=True)
        return
    h = df.text.map(text_hash)
    train_h = set(h[df.split == "train"])
    for s in sorted(set(df.split) - {"train"}):
        n = int(h[df.split == s].isin(train_h).sum())
        if n:
            raise SystemExit(f"LEAKAGE: {n} {s} texts also occur in train; rebuild the dataset")
    cache.write_text(json.dumps({"sha256": ds_sha, "result": "no train/eval text overlap"}))


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=str(Path(__file__).resolve().parents[1] / "configs" / "ai_detector.yaml"))
    ap.add_argument("--set", nargs="*", default=[])
    ap.add_argument("--dataset", default=str(PROCESSED_DIR / "dataset.parquet"))
    args = ap.parse_args()

    cfg = yaml.safe_load(Path(args.config).read_text())
    cfg.update(parse_overrides(args.set))
    ds_path = Path(args.dataset)
    df = load_dataset(ds_path)
    ds_sha = file_sha256(ds_path)
    check_text_leakage(df, ds_path, ds_sha)

    train_df = df[df.split == "train"]
    val_df = df[df.split == "validation"]
    train_df, removed = apply_ablation(train_df, cfg.get("ablation") or {})
    train_df = stratified_subset(train_df, cfg.get("max_train_rows"), cfg["seed"])
    val_df = stratified_subset(val_df, cfg.get("max_val_rows"), cfg["seed"])

    report_path = MANIFEST_DIR / "dataset_report.json"
    provenance = {
        "dataset_file": ds_path.name,
        "dataset_sha256": ds_sha,
        "dataset_report_sha256": file_sha256(report_path) if report_path.exists() else None,
        "train_rows": len(train_df), "validation_rows": len(val_df),
        "train_row_ids_sha256": row_ids_digest(train_df), "validation_row_ids_sha256": row_ids_digest(val_df),
        "ablation_removed_rows": removed,
        "is_subset": cfg.get("max_train_rows") is not None or cfg.get("max_val_rows") is not None,
        "train_rows_by_source": train_df.source_dataset.value_counts().to_dict(),
    }
    run_dir = ARTIFACT_DIR / "runs" / cfg["run_name"]
    print(json.dumps({"run_dir": str(run_dir), **provenance}, indent=2), flush=True)
    meta = train(cfg, train_df, val_df, run_dir, provenance)
    if "interrupted_at_step" in meta:
        print(f"Stopped at step {meta['interrupted_at_step']}; re-run the same command to resume.")
    else:
        print(json.dumps({"best": meta["best"], "steps": meta["optimizer_steps_done"]}, indent=2))


if __name__ == "__main__":
    main()


### `ml/scripts/calibrate_thresholds.py`
Fit temperature scaling and FPR-constrained thresholds on VALIDATION logits only.

In [ ]:
%%writefile ml/scripts/calibrate_thresholds.py
"""Fit temperature scaling and FPR-constrained thresholds on VALIDATION logits only.

Usage: python ml/scripts/calibrate_thresholds.py --run deberta_multitask_s1 [--operating-point fpr_0.01]
Writes <run>/best/calibration.json.
"""
from __future__ import annotations

import argparse
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scriptsense_ml.evaluation.calibration import apply_temperature, fit_temperature, select_thresholds  # noqa: E402
from scriptsense_ml.evaluation.metrics import MIN_EXPECTED_FALSE_POSITIVES, expected_calibration_error  # noqa: E402
from scriptsense_ml.paths import ARTIFACT_DIR  # noqa: E402

TARGETS = (0.01, 0.05)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True)
    ap.add_argument("--operating-point", default="auto",
                    help="fpr_0.01 | fpr_0.05 | auto (strictest target with enough human validation rows)")
    args = ap.parse_args()
    best = ARTIFACT_DIR / "runs" / args.run / "best"
    z = np.load(best / "val_logits.npz", allow_pickle=True)
    l4, l2, y4, y2 = z["logits4"], z["logits2"], z["y4"], z["y2"]

    t4, t2 = fit_temperature(l4, y4), fit_temperature(l2, y2)
    p_ai = apply_temperature(l2, t2)[:, 1]
    thresholds = select_thresholds(p_ai[y2 != -1], y2[y2 != -1], TARGETS)

    if args.operating_point == "auto":
        ok = [k for k in (f"fpr_{t}" for t in TARGETS)
              if thresholds[k]["expected_false_positives_at_target"] >= MIN_EXPECTED_FALSE_POSITIVES]
        op = ok[0] if ok else None
    else:
        op = args.operating_point
    if op is None:
        raise SystemExit("Not enough human validation rows to support any declared FPR operating point.")

    m4, m2 = y4 != -1, y2 != -1
    ece = {
        "four_way_before": expected_calibration_error(apply_temperature(l4[m4], 1.0), y4[m4])[0],
        "four_way_after": expected_calibration_error(apply_temperature(l4[m4], t4), y4[m4])[0],
        "binary_before": expected_calibration_error(apply_temperature(l2[m2], 1.0), y2[m2])[0],
        "binary_after": expected_calibration_error(apply_temperature(l2[m2], t2), y2[m2])[0],
        "note": "in-sample on the validation set used for fitting; held-out calibration is in evaluate_detector output",
    }
    out = {
        "fitted_at": datetime.now(timezone.utc).isoformat(),
        "fitted_on": "validation split only",
        "temperature_4way": t4, "temperature_binary": t2,
        "thresholds": thresholds,
        "deployed_operating_point": op,
        "deployed_threshold": thresholds[op]["threshold"],
        "validation_ece": ece,
        "n_validation": int(len(y2)),
    }
    (best / "calibration.json").write_text(json.dumps(out, indent=2))
    print(json.dumps(out, indent=2))


if __name__ == "__main__":
    main()


### `ml/scripts/evaluate_detector.py`
Evaluate a trained, calibrated run on held-out sets. Never used for selection or thresholds.

In [ ]:
%%writefile ml/scripts/evaluate_detector.py
"""Evaluate a trained, calibrated run on held-out sets. Never used for selection or thresholds.

Usage: python ml/scripts/evaluate_detector.py --run deberta_multitask_s1 [--sets semeval_a/test txd22/test raid/challenge]
       [--max-rows N]   (documented subsample per set, for smoke runs only)

Every invocation on a final set is appended to <run>/final_eval_audit.jsonl so repeated
peeking at test data is visible.
"""
from __future__ import annotations

import argparse
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, roc_auc_score
from transformers import AutoTokenizer

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scriptsense_ml.data.io import file_sha256  # noqa: E402
from scriptsense_ml.data.schema import CLASS4_NAMES, UNKNOWN  # noqa: E402
from scriptsense_ml.evaluation.calibration import apply_temperature  # noqa: E402
from scriptsense_ml.evaluation.metrics import (  # noqa: E402
    binary_metrics, bootstrap_ci, four_class_metrics, length_bucket, recall_by_class_at_threshold,
)
from scriptsense_ml.inference.predict import inference_amp_dtype, predict_logits  # noqa: E402
from scriptsense_ml.models.multitask import MultiTaskDetector  # noqa: E402
from scriptsense_ml.paths import ARTIFACT_DIR, PROCESSED_DIR  # noqa: E402
from scriptsense_ml.training.data import load_dataset  # noqa: E402

DEFAULT_SETS = ["semeval_a/test", "txd22/test", "raid/challenge"]
SLICES = ["domain", "generator", "attack_or_edit_type", "length_bucket"]


def _macro_f1(y4, p4):
    m = y4 != UNKNOWN
    return f1_score(y4[m], p4[m].argmax(1), average="macro")


def _fpr(y2, pai, t):
    neg = y2 == 0
    if not neg.any():
        raise ValueError("no negatives")
    return float((pai[neg] >= t).mean())


def evaluate_set(df, p4, pai, cal) -> dict:
    t = cal["deployed_threshold"]
    y4, y2 = df.class_4way.to_numpy(), df.label_binary.to_numpy()
    groups = (df.prompt_id.where(df.prompt_id != "", df.original_document_id)
              .where(lambda s: s != "", df.row_id)).to_numpy()
    res = {"n": len(df), "four_way": four_class_metrics(p4, y4),
           "binary": binary_metrics(pai, y2, threshold=t),
           "binary_flag_rate_by_true_class": recall_by_class_at_threshold(pai, y4, t),
           "deployed_operating_point": cal["deployed_operating_point"], "threshold": t,
           "ci95_grouped_bootstrap": {}}
    m4 = y4 != UNKNOWN
    if m4.sum() and len(np.unique(y4[m4])) > 1:
        res["ci95_grouped_bootstrap"]["macro_f1_4way"] = bootstrap_ci(_macro_f1, (y4, p4), 500, 0, groups)
    m2 = y2 != UNKNOWN
    if len(np.unique(y2[m2])) == 2:
        res["ci95_grouped_bootstrap"]["auroc"] = bootstrap_ci(
            roc_auc_score, (y2[m2], pai[m2]), 500, 0, groups[m2])
        res["ci95_grouped_bootstrap"]["human_fpr"] = bootstrap_ci(
            lambda a, b: _fpr(a, b, t), (y2[m2], pai[m2]), 500, 0, groups[m2])
    df = df.assign(length_bucket=length_bucket(df.text.str.split().str.len().to_numpy()))
    res["slices"] = {}
    for col in SLICES:
        res["slices"][col] = {}
        for val, idx in df.groupby(col).indices.items():
            if len(idx) < 30:
                continue
            b = binary_metrics(pai[idx], y2[idx], threshold=t)
            f = four_class_metrics(p4[idx], y4[idx])
            res["slices"][col][str(val) or "unknown"] = {
                "n": int(len(idx)), "human_fpr": b.get("human_false_positive_rate"), "ai_recall": b.get("ai_recall"),
                "auroc": b.get("auroc"), "macro_f1_4way": f.get("macro_f1_present_classes"), "accuracy_4way": f.get("accuracy")}
    return res


def plot(res: dict, name: str, out_dir: Path) -> None:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    cm = np.array(res["four_way"]["confusion_matrix"]["matrix"])
    axes[0].imshow(cm, cmap="Blues")
    short = ["HUMAN", "AI_GEN", "AI_PARA", "MIXED"]
    axes[0].set_xticks(range(4), short, rotation=30)
    axes[0].set_yticks(range(4), short)
    axes[0].set_xlabel("predicted")
    axes[0].set_ylabel("true")
    for i in range(4):
        for j in range(4):
            axes[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
    axes[0].set_title(f"{name}: 4-way confusion")
    bins = res["binary"].get("reliability_bins", [])
    if bins:
        axes[1].plot([0, 1], [0, 1], "--", color="grey")
        axes[1].plot([b["confidence"] for b in bins], [b["accuracy"] for b in bins], "o-")
        axes[1].set_xlabel("confidence")
        axes[1].set_ylabel("accuracy")
        axes[1].set_title("binary reliability")
    fig.tight_layout()
    fig.savefig(out_dir / f"{name.replace('/', '_')}.png", dpi=120)
    plt.close(fig)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True)
    ap.add_argument("--sets", nargs="*", default=DEFAULT_SETS)
    ap.add_argument("--max-rows", type=int, default=None)
    ap.add_argument("--dataset", default=str(PROCESSED_DIR / "dataset.parquet"))
    args = ap.parse_args()

    run = ARTIFACT_DIR / "runs" / args.run
    best = run / "best"
    cal = json.loads((best / "calibration.json").read_text())
    meta = json.loads((best / "model_meta.json").read_text())
    max_len = meta["config"]["max_length"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = inference_amp_dtype(device)
    model = MultiTaskDetector.load(best).to(device)
    tok = AutoTokenizer.from_pretrained(best / "tokenizer")

    df = load_dataset(Path(args.dataset))
    out_dir = run / "eval"
    out_dir.mkdir(exist_ok=True)
    summary = []
    for name in args.sets:
        src, split = name.split("/")
        if split == "train":
            raise SystemExit("refusing to report training-split metrics as held-out results")
        part = df[(df.source_dataset == src) & (df.split == split)]
        subsampled = args.max_rows is not None and len(part) > args.max_rows
        if subsampled:
            part = part.sample(n=args.max_rows, random_state=0)
        if part.empty:
            print(f"skip {name}: no rows")
            continue
        l4, l2 = predict_logits(model, tok, part.text.tolist(), max_len, 32, device, amp)
        p4 = apply_temperature(l4, cal["temperature_4way"])
        pai = apply_temperature(l2, cal["temperature_binary"])[:, 1]
        res = evaluate_set(part.reset_index(drop=True), p4, pai, cal)
        res.update({"set": name, "subsampled_to": args.max_rows if subsampled else None,
                    "evaluated_at": datetime.now(timezone.utc).isoformat(),
                    "dataset_sha256": file_sha256(Path(args.dataset)), "passage_level": True,
                    "note": f"passage-level: each row truncated to {max_len} tokens"})
        if src == "raid" and split == "challenge":
            unseen = part.generator.isin(["cohere", "cohere-chat"]).to_numpy()
            res["unseen_generator_subset"] = {
                "binary": binary_metrics(pai[unseen], part.label_binary.to_numpy()[unseen], cal["deployed_threshold"])}
            res["unseen_generator_subset"]["binary"].pop("reliability_bins", None)
        (out_dir / f"{name.replace('/', '_')}.json").write_text(json.dumps(res, indent=2, default=str))
        plot(res, name, out_dir)
        b, f = res["binary"], res["four_way"]
        row = {"set": name, "n": res["n"], "subsampled": subsampled,
               "macro_f1_4way": f.get("macro_f1_present_classes"), "auroc": b.get("auroc"),
               "auprc": b.get("auprc"), "human_fpr": b.get("human_false_positive_rate"),
               "ai_recall": b.get("ai_recall"), "brier_binary": b.get("brier"), "ece_binary": b.get("ece"),
               "ece_4way": f.get("ece")}
        for cname in CLASS4_NAMES[1:]:
            row[f"recall_{cname}"] = f.get("per_class", {}).get(cname, {}).get("recall")
        summary.append(row)
        with open(run / "final_eval_audit.jsonl", "a") as fh:
            fh.write(json.dumps({"set": name, "at": res["evaluated_at"], "n": res["n"]}) + "\n")
        print(json.dumps(row, default=str))
    pd.DataFrame(summary).to_csv(out_dir / "metrics_summary.csv", index=False)


if __name__ == "__main__":
    main()


## 2. Code: Tests

### `ml/tests/test_data_layer.py`

In [ ]:
%%writefile ml/tests/test_data_layer.py
import numpy as np
import pytest

from scriptsense_ml.data import label_mapping as lm
from scriptsense_ml.data.deduplication import MinHasher, exact_duplicate_groups, near_duplicate_pairs
from scriptsense_ml.data.normalization import (
    decode_mixed_lines, dedup_key, normalize_for_model, text_hash,
)
from scriptsense_ml.data.schema import UNKNOWN, Class4, binary_from_class4, validate_row
from scriptsense_ml.data.splitting import (
    LeakageError, assert_no_leakage, assign_splits, connected_components, leakage_report,
)


# ---------------------------------------------------------------- normalization
def test_normalize_for_model_strips_invisible_but_keeps_case_and_paragraphs():
    raw = "Hello​  World!\r\n\r\n\r\n  Next para﻿"
    assert normalize_for_model(raw) == "Hello World!\n\nNext para"


def test_dedup_key_is_case_punct_and_homoglyph_width_insensitive():
    assert dedup_key("Ｈello, WORLD!!") == dedup_key("hello world")
    assert text_hash("A  b.c") == text_hash("a b c")


def test_decode_mixed_lines_handles_utf8_and_cp1252():
    raw = "café utf8\n".encode("utf-8") + b"it\x92s cp1252\n" + b"\xef\xbb\xbfx"
    text, stats = decode_mixed_lines(b"\xef\xbb\xbf" + raw)
    assert text.splitlines()[:2] == ["café utf8", "it’s cp1252"]
    assert stats["cp1252"] == 1 and stats["latin1_fallback"] == 0


# ---------------------------------------------------------------- labels
def test_binary_from_class4():
    assert binary_from_class4(Class4.HUMAN_ONLY) == 0
    for c in (Class4.AI_GENERATED, Class4.AI_PARAPHRASED_OR_REFINED, Class4.MIXED_OR_AI_ASSISTED):
        assert binary_from_class4(c) == 1
    assert binary_from_class4(UNKNOWN) == UNKNOWN


def test_semeval_mapping():
    assert lm.map_semeval_a(0)[:2] == (Class4.HUMAN_ONLY, 0)
    assert lm.map_semeval_a(1)[:2] == (Class4.AI_GENERATED, 1)
    with pytest.raises(ValueError):
        lm.map_semeval_a(2)


@pytest.mark.parametrize("model,attack,expected", [
    ("human", "none", (Class4.HUMAN_ONLY, 0)),
    ("gpt4", "none", (Class4.AI_GENERATED, 1)),
    ("gpt4", "paraphrase", (Class4.AI_PARAPHRASED_OR_REFINED, 1)),
    ("human", "paraphrase", (Class4.MIXED_OR_AI_ASSISTED, 1)),
    # character-level perturbations are never "humanized"
    ("mistral", "homoglyph", (Class4.AI_GENERATED, 1)),
    ("mistral", "zero_width_space", (Class4.AI_GENERATED, 1)),
    ("human", "whitespace", (Class4.HUMAN_ONLY, 0)),
    # synonym swaps: AI involvement known, 4-way label unsupported
    ("chatgpt", "synonym", (UNKNOWN, 1)),
    ("human", "synonym", (UNKNOWN, UNKNOWN)),
])
def test_raid_mapping(model, attack, expected):
    assert lm.map_raid(model, attack)[:2] == expected


def test_raid_unknown_attack_raises():
    with pytest.raises(ValueError):
        lm.map_raid("gpt4", "new_attack")


def test_txd22_mapping_covers_all_22_published_labels():
    published = [
        "ChatGPT", "Grok", "Copilot", "Gemini", "DeepSeek", "DeepAI", "Qwen", "Claude", "Llama",
        "Gemma", "Z.ai", "Perplexity", "Mistral AI", "Poe", "Nova", "BLACKBOXAI", "Pi",
        "Human written",
        "AI-generated and Human-written",  # the raw file uses a non-breaking space here
        "AI-generated and AI-refined",
        "Human-written and AI-refined",
        "AI-generated and AI-refined and Human-written",
    ]
    got = {s: lm.map_txd22(s)[0] for s in published}
    assert sum(v == Class4.AI_GENERATED for v in got.values()) == 17
    assert got["Human written"] == Class4.HUMAN_ONLY
    assert got["AI-generated and AI-refined"] == Class4.AI_PARAPHRASED_OR_REFINED
    assert got["AI-generated and Human-written"] == Class4.MIXED_OR_AI_ASSISTED
    assert got["Human-written and AI-refined"] == Class4.MIXED_OR_AI_ASSISTED
    assert got["AI-generated and AI-refined and Human-written"] == Class4.MIXED_OR_AI_ASSISTED
    with pytest.raises(ValueError):
        lm.map_txd22("Some new model")


def _valid_row(**kw):
    row = dict(row_id="x:1", text="hello", raw_text="hello", source_dataset="x", source_row_id="1",
               original_document_id="", prompt_id="", generator="", domain="", attack_or_edit_type="none",
               provenance="", source_label="", class_4way=0, label_binary=0, split="", split_fixed=False,
               group_keys="", license_reference="")
    row.update(kw)
    return row


def test_validate_row_rejects_inconsistent_labels():
    validate_row(_valid_row())
    validate_row(_valid_row(class_4way=UNKNOWN, label_binary=1))
    with pytest.raises(ValueError):
        validate_row(_valid_row(class_4way=2, label_binary=0))
    with pytest.raises(ValueError):
        validate_row(_valid_row(class_4way=UNKNOWN, label_binary=UNKNOWN))


# ---------------------------------------------------------------- dedup
BASE = ("The committee reviewed the proposal in detail and concluded that the budget "
        "allocation for the next fiscal year should prioritise infrastructure repair, "
        "public transport, and the expansion of community health clinics in rural areas.")


def test_exact_duplicates():
    hashes = [text_hash(t) for t in [BASE, BASE.upper(), "other text entirely"]]
    assert exact_duplicate_groups(hashes) == {hashes[0]: [0, 1]}


def test_minhash_is_deterministic_and_finds_near_duplicates():
    near = BASE.replace("in detail", "in great detail")
    other = "Photosynthesis converts light energy into chemical energy stored in glucose molecules."
    mh = MinHasher(num_perm=128, seed=7)
    sigs = np.stack([mh.signature(t) for t in (BASE, near, other)])
    assert np.array_equal(sigs, np.stack([MinHasher(128, 7).signature(t) for t in (BASE, near, other)]))
    pairs = near_duplicate_pairs(sigs, bands=32, threshold=0.6)
    assert [(i, j) for i, j, _ in pairs] == [(0, 1)]


# ---------------------------------------------------------------- splitting
def test_components_join_rows_sharing_any_key():
    keys = [["p:1"], ["p:1", "dup:a"], ["dup:a"], ["p:2"]]
    comps = connected_components(keys)
    assert comps[0] == comps[1] == comps[2] != comps[3]


def test_assign_splits_respects_pins_and_groups():
    keys = [["p:1"], ["p:1"], ["p:2"], ["p:2"], ["p:3"]]
    comps = connected_components(keys)
    names = {c: f"c{c}" for c in set(comps)}
    fixed = ["", "validation", "train", "", ""]
    splits, conflicts = assign_splits(comps, names, fixed, {"train": 0.8, "validation": 0.1, "test": 0.1}, 0)
    assert splits[0] == splits[1] == "validation"   # evaluation pin wins for the whole group
    assert splits[2] == splits[3] == "train"
    assert not conflicts
    assert_no_leakage(splits, keys)


def test_assign_splits_reports_conflicting_eval_pins():
    keys = [["p:1"], ["p:1"]]
    comps = connected_components(keys)
    splits, conflicts = assign_splits(comps, {0: "a"}, ["validation", "test"], {"train": 1.0}, 0)
    assert splits == ["", ""] and conflicts


def test_assign_splits_is_reproducible_and_roughly_proportional():
    keys = [[f"p:{i}"] for i in range(5000)]
    comps = connected_components(keys)
    names = {c: f"name{c}" for c in comps}
    ratios = {"train": 0.8, "validation": 0.1, "test": 0.1}
    s1, _ = assign_splits(comps, names, [""] * 5000, ratios, seed=42)
    s2, _ = assign_splits(comps, names, [""] * 5000, ratios, seed=42)
    assert s1 == s2
    assert 0.77 < s1.count("train") / 5000 < 0.83


def test_leakage_detected():
    keys = [["p:1"], ["p:1"]]
    assert leakage_report(["train", "test"], keys)["num_leaking_keys"] == 1
    with pytest.raises(LeakageError):
        assert_no_leakage(["train", "test"], keys)


def test_unique_trigram_ratio_flags_loops_only():
    from scriptsense_ml.data.normalization import unique_trigram_ratio
    loop = "it is a film that is not for everyone because " * 20
    assert unique_trigram_ratio(loop) < 0.1
    assert unique_trigram_ratio(BASE + " " + BASE.replace("committee", "council")) > 0.5
    assert unique_trigram_ratio("too short to judge") == 1.0


### `ml/tests/test_build_pipeline.py`
End-to-end smoke test of deduplicate_and_split on a tiny SYNTHETIC fixture.

In [ ]:
%%writefile ml/tests/test_build_pipeline.py
"""End-to-end smoke test of deduplicate_and_split on a tiny SYNTHETIC fixture.

The fixture is hand-written test data, isolated in a temp dir; it is never mixed into
real datasets or benchmark results.
"""
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import pytest

from scriptsense_ml.data.io import rows_to_frame

ML = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ML / "scripts"))
import prepare_raid  # noqa: E402

SYN = "SYNTHETIC TEST FIXTURE - not real data"
TOPICS = ["rivers", "volcanoes", "bridges", "orchards", "satellites", "glaciers", "markets", "libraries",
          "harbors", "deserts", "forests", "railways"]


def sentence(topic, k, variant=0):
    return (f"Paragraph {k} about {topic}: the regional survey of {topic} recorded measurement {k * 7 + variant} "
            f"and observers noted that {topic} in district {k} changed after season {k + 3}, while volunteers "
            f"catalogued {k * 11} distinct features near the {topic} station number {k * 13}.")


def row(src, rid, text, c4, b, split="", fixed=False, keys="", prompt=""):
    return dict(row_id=f"{src}:{rid}", text=text, raw_text=text, source_dataset=src, source_row_id=rid,
                original_document_id="", prompt_id=prompt, generator="synthetic", domain="synthetic",
                attack_or_edit_type="none", provenance=SYN, source_label=str(c4), class_4way=c4,
                label_binary=b, split=split, split_fixed=fixed, group_keys=keys or f"{src}:doc:{rid}",
                license_reference=SYN)


@pytest.fixture()
def fixture_dirs(tmp_path):
    data, manifests = tmp_path / "data", tmp_path / "manifests"
    (data / "processed").mkdir(parents=True)
    manifests.mkdir()
    sem = [row("semeval_a", f"tr{i}", sentence(TOPICS[i % 12], i), i % 2, i % 2, "train", True) for i in range(40)]
    sem += [row("semeval_a", f"va{i}", sentence(TOPICS[i % 12], 100 + i), i % 2, i % 2, "validation", True)
            for i in range(10)]
    sem += [row("semeval_a", f"te{i}", sentence(TOPICS[i % 12], 200 + i), i % 2, i % 2, "test", True)
            for i in range(10)]
    # exact duplicate of a test row placed in train -> must be removed from training
    sem.append(row("semeval_a", "trdup", sentence(TOPICS[0], 200), 0, 0, "train", True))
    # same text, conflicting label -> all copies dropped
    sem.append(row("semeval_a", "trconf", sentence(TOPICS[1], 201), 0, 0, "train", True))
    txd = []
    for p in range(30):
        for c in range(4):
            txd.append(row("txd22", f"{p}-{c}", sentence(TOPICS[c], 1000 + p * 4 + c, c), c, int(c > 0),
                           keys=f"txd22:prompt:{p}", prompt=str(p)))
    rows_to_frame(sem).to_parquet(data / "processed" / "semeval_a.parquet", index=False)
    rows_to_frame(txd).to_parquet(data / "processed" / "txd22.parquet", index=False)
    return data, manifests


def test_build_pipeline_end_to_end(fixture_dirs):
    data, manifests = fixture_dirs
    env = {**os.environ, "SCRIPTSENSE_DATA_DIR": str(data), "SCRIPTSENSE_MANIFEST_DIR": str(manifests)}
    res = subprocess.run([sys.executable, str(ML / "scripts" / "deduplicate_and_split.py")],
                         env=env, capture_output=True, text=True, timeout=600)
    assert res.returncode == 0, res.stderr[-3000:]
    out = pd.read_parquet(data / "processed" / "dataset.parquet")
    report = json.loads((manifests / "dataset_report.json").read_text())

    assert "semeval_a:trdup" not in set(out["row_id"])              # test copy kept, train copy collapsed
    assert out.loc[out.row_id == "semeval_a:te0", "split"].item() == "test"
    assert not out["row_id"].isin(["semeval_a:trconf", "semeval_a:te1"]).any()  # conflicting labels dropped
    assert report["dropped"]["exact_dup_conflicting_labels"] == 2
    # official splits preserved
    assert set(out.loc[out.row_id.str.startswith("semeval_a:va"), "split"]) == {"validation"}
    # every TXD prompt lives in exactly one split
    txd = out[out.source_dataset == "txd22"]
    assert (txd.groupby("prompt_id")["split"].nunique() == 1).all()
    assert report["leakage"]["num_leaking_keys"] == 0


def test_raid_category_and_water_fill():
    assert prepare_raid.category("human", "none") == "human_clean"
    assert prepare_raid.category("human", "paraphrase") is None
    assert prepare_raid.category("gpt4", "paraphrase") == "ai_paraphrase"
    assert prepare_raid.category("gpt4", "homoglyph") == "ai_other"
    alloc = prepare_raid.water_fill({"a": 2, "b": 100, "c": 100}, 50)
    assert alloc["a"] == 2 and alloc["b"] + alloc["c"] == 48 and sum(alloc.values()) == 50
    assert sum(prepare_raid.water_fill({"a": 3, "b": 4}, 50).values()) == 7  # never pads


def test_raid_holdout_generator_never_in_train_pool():
    cfg = {"seed": 1, "raid": {"challenge_lineage_fraction": 0.1, "validation_lineage_fraction": 0.05,
                               "holdout_generators": ["cohere"]}}

    class R:  # minimal row stub
        def __init__(self, model, sid):
            self.model, self.attack, self.domain, self.source_id = model, "none", "news", sid

    pools = [prepare_raid.classify(R("cohere", f"s{i}"), cfg) for i in range(500)]
    assert all(p is None or p[0] == "challenge" for p in pools)
    assert any(p is not None for p in pools)


### `ml/tests/test_model_training.py`

In [ ]:
%%writefile ml/tests/test_model_training.py
import json

import numpy as np
import pandas as pd
import pytest
import torch
from transformers import DebertaV2Config

from scriptsense_ml.data.schema import UNKNOWN
from scriptsense_ml.evaluation.calibration import apply_temperature, fit_temperature, select_thresholds
from scriptsense_ml.evaluation.metrics import (
    binary_metrics, bootstrap_ci, expected_calibration_error, four_class_metrics, threshold_for_fpr,
)
from scriptsense_ml.models.multitask import MultiTaskDetector, class_weights_from_counts, multitask_loss
from scriptsense_ml.training.data import apply_ablation, stratified_subset

TOKENIZER = "microsoft/deberta-v3-base"


def tiny_config():
    return DebertaV2Config(vocab_size=128100, hidden_size=32, num_hidden_layers=2, num_attention_heads=2,
                           intermediate_size=64, max_position_embeddings=512, relative_attention=True,
                           position_buckets=256, pos_att_type=["p2c", "c2p"], norm_rel_ebd="layer_norm",
                           max_relative_positions=-1, share_att_key=True, type_vocab_size=0)


# ------------------------------------------------------------------ loss
def test_unknown_labels_do_not_train_the_four_way_head():
    l4 = torch.randn(3, 4, requires_grad=True)
    l2 = torch.randn(3, 2, requires_grad=True)
    y4 = torch.tensor([UNKNOWN, UNKNOWN, UNKNOWN])
    y2 = torch.tensor([1, 0, 1])
    loss, parts = multitask_loss(l4, l2, y4, y2)
    loss.backward()
    assert parts["loss_4way"] == 0.0
    assert torch.all(l4.grad == 0)
    assert l2.grad.abs().sum() > 0


def test_binary_only_mode_ignores_four_way_head():
    l4 = torch.randn(2, 4, requires_grad=True)
    loss, _ = multitask_loss(l4, torch.randn(2, 2), torch.tensor([0, 1]), torch.tensor([0, 1]), four_way_weight=0.0)
    loss.backward()
    assert torch.all(l4.grad == 0)


def test_class_weights():
    w = class_weights_from_counts({0: 100, 1: 10000, 3: 400}, power=0.5)
    assert w[2] == 0 and w[0] > w[3] > w[1]
    assert torch.isclose(w[w > 0].mean(), torch.tensor(1.0))


# ------------------------------------------------------------------ calibration / metrics
def test_temperature_scaling_recovers_overconfidence():
    rng = np.random.default_rng(0)
    y = rng.integers(0, 4, 4000)
    true_logits = rng.normal(size=(4000, 4))
    true_logits[np.arange(4000), y] += 1.5
    # sample labels from the true distribution, then present over-confident logits (x3)
    p = apply_temperature(true_logits, 1.0)
    y = np.array([rng.choice(4, p=pi) for pi in p])
    t = fit_temperature(true_logits * 3.0, y)
    assert 2.5 < t < 3.5


def test_threshold_for_fpr_controls_false_positive_rate():
    neg = np.linspace(0, 1, 1000)
    for f in (0.01, 0.05, 0.2):
        t = threshold_for_fpr(neg, f)
        assert (neg >= t).mean() <= f + 1e-9
    thr = select_thresholds(np.r_[neg, np.ones(10)], np.r_[np.zeros(1000), np.ones(10)])
    assert thr["fpr_0.01"]["validation_fpr"] <= 0.01 and thr["fpr_0.01"]["n_human_validation"] == 1000


def test_metrics_exclude_unknown_and_report_classes():
    p4 = np.eye(4)[[0, 1, 2, 3, 1]]
    y4 = np.array([0, 1, 2, UNKNOWN, 2])
    m = four_class_metrics(p4, y4)
    assert m["n"] == 4 and m["per_class"]["AI_PARAPHRASED_OR_REFINED"]["recall"] == 0.5
    b = binary_metrics(np.array([0.1, 0.9, 0.8, 0.2]), np.array([0, 1, 1, UNKNOWN]))
    assert b["n"] == 3 and b["human_false_positive_rate"] == 0.0 and b["auroc"] == 1.0
    ece, _ = expected_calibration_error(np.array([[0.0, 1.0], [1.0, 0.0]]), np.array([1, 0]))
    assert ece == 0.0


def test_grouped_bootstrap_ci_contains_point_estimate():
    rng = np.random.default_rng(1)
    y = rng.integers(0, 2, 600)
    p = np.clip(y * 0.6 + rng.normal(0.2, 0.2, 600), 0, 1)
    from sklearn.metrics import roc_auc_score
    ci = bootstrap_ci(roc_auc_score, (y, p), n_boot=200, groups=np.arange(600) // 3)
    assert ci["low"] <= roc_auc_score(y, p) <= ci["high"]


# ------------------------------------------------------------------ data selection
def _df(n=200):
    return pd.DataFrame({"row_id": [f"r{i}" for i in range(n)], "text": ["t"] * n,
                         "source_dataset": ["raid"] * (n // 2) + ["txd22"] * (n // 2),
                         "attack_or_edit_type": ["paraphrase", "none"] * (n // 2),
                         "class_4way": [2, 1, 3, 0] * (n // 4), "label_binary": [1, 1, 1, 0] * (n // 4)})


def test_ablation_filters():
    df = _df()
    out, removed = apply_ablation(df, {"exclude_raid_paraphrase": True, "exclude_txd22_mixed": True})
    assert removed["raid_paraphrase"] == 50
    assert not ((out.source_dataset == "raid") & (out.attack_or_edit_type == "paraphrase")).any()
    assert not ((out.source_dataset == "txd22") & out.class_4way.isin([2, 3])).any()


def test_stratified_subset_is_deterministic_and_proportional():
    df = _df(400)
    a, b = stratified_subset(df, 100, 3), stratified_subset(df, 100, 3)
    assert a.row_id.tolist() == b.row_id.tolist() and 90 <= len(a) <= 110
    assert stratified_subset(df, None, 3) is df


# ------------------------------------------------------------------ training smoke (CPU, tiny random encoder)
def test_training_smoke_tiny_encoder(tmp_path, monkeypatch):
    from scriptsense_ml.training import trainer

    real_init = MultiTaskDetector.__init__

    def tiny_init(self, encoder_name, dropout=0.1, encoder_config=None):
        real_init(self, encoder_name, dropout, encoder_config=tiny_config())

    monkeypatch.setattr(MultiTaskDetector, "__init__", tiny_init)
    rng = np.random.default_rng(0)
    words = ["alpha", "beta", "gamma", "delta", "river", "stone", "light", "paper"]

    def make(n, prefix):
        return pd.DataFrame({
            "row_id": [f"{prefix}{i}" for i in range(n)],
            "text": [" ".join(rng.choice(words, 20)) for _ in range(n)],
            "class_4way": [[0, 1, 2, 3, UNKNOWN][i % 5] for i in range(n)],
            "label_binary": [[0, 1, 1, 1, 1][i % 5] for i in range(n)],
            "source_dataset": "synthetic_fixture"})

    cfg = dict(run_name="smoke", model_name="tiny", tokenizer_name=TOKENIZER, mode="multitask", seed=1,
               max_length=32, learning_rate=1e-3, epochs=2, per_device_batch_size=4, grad_accum_steps=2,
               weight_decay=0.01, warmup_ratio=0.1, max_grad_norm=1.0, binary_loss_weight=0.5,
               class_weight_power=0.5, precision="fp32", gradient_checkpointing=False,
               early_stopping_patience=5, eval_every_steps=None, selection_metric="auto", eval_batch_size=8)
    meta = trainer.train(cfg, make(40, "t"), make(10, "v"), tmp_path / "run", {"fixture": "synthetic"})
    best = tmp_path / "run" / "best"
    assert meta["best"] is not None and (best / "heads.pt").exists()
    assert json.loads((best / "model_meta.json").read_text())["label_map_4way"]["2"] == "AI_PARAPHRASED_OR_REFINED"
    logits = np.load(best / "val_logits.npz")
    assert logits["logits4"].shape == (10, 4) and logits["logits2"].shape == (10, 2)
    reloaded = MultiTaskDetector.load(best)
    assert reloaded.head4.out_features == 4
    log = [json.loads(line) for line in open(tmp_path / "run" / "train_log.jsonl")]
    assert any(r["event"] == "eval" for r in log)


def _smoke_cfg(**kw):
    cfg = dict(run_name="smoke", model_name="tiny", tokenizer_name=TOKENIZER, mode="multitask", seed=1,
               max_length=32, learning_rate=1e-3, epochs=2, per_device_batch_size=4, grad_accum_steps=2,
               weight_decay=0.01, warmup_ratio=0.1, max_grad_norm=1.0, binary_loss_weight=0.5,
               class_weight_power=0.5, precision="fp32", gradient_checkpointing=False,
               early_stopping_patience=5, eval_every_steps=None, selection_metric="auto", eval_batch_size=8)
    cfg.update(kw)
    return cfg


def _frames():
    rng = np.random.default_rng(0)
    words = ["alpha", "beta", "gamma", "delta", "river", "stone", "light", "paper"]

    def make(n, prefix):
        return pd.DataFrame({
            "row_id": [f"{prefix}{i}" for i in range(n)],
            "text": [" ".join(rng.choice(words, 20)) for _ in range(n)],
            "class_4way": [[0, 1, 2, 3, UNKNOWN][i % 5] for i in range(n)],
            "label_binary": [[0, 1, 1, 1, 1][i % 5] for i in range(n)],
            "source_dataset": "synthetic_fixture"})
    return make(40, "t"), make(10, "v")


def test_interrupted_and_resumed_training_matches_uninterrupted(tmp_path, monkeypatch):
    """Checkpoint/resume must be exact: same final weights as a run that never stopped."""
    from scriptsense_ml.training import trainer

    real_init = MultiTaskDetector.__init__
    monkeypatch.setattr(MultiTaskDetector, "__init__",
                        lambda self, n, dropout=0.1, encoder_config=None: real_init(self, n, dropout, tiny_config()))
    tr, va = _frames()
    prov = {"train_row_ids_sha256": "abc"}

    trainer.train(_smoke_cfg(), tr, va, tmp_path / "full", prov)
    # interrupt mid-epoch-1 (step 7 of 10, epoch 0 has 5 steps), then resume with the same command
    out = trainer.train(_smoke_cfg(checkpoint_every_steps=3, stop_after_steps=7), tr, va, tmp_path / "res", prov)
    assert out["interrupted_at_step"] == 7 and (tmp_path / "res" / "last_checkpoint.pt").exists()
    meta = trainer.train(_smoke_cfg(checkpoint_every_steps=3), tr, va, tmp_path / "res", prov)
    assert meta["optimizer_steps_done"] == 10 and len(meta["resumed_at"]) == 1
    assert not (tmp_path / "res" / "last_checkpoint.pt").exists()  # cleaned up after finishing

    a = torch.load(tmp_path / "full" / "best" / "heads.pt", weights_only=True)
    b = torch.load(tmp_path / "res" / "best" / "heads.pt", weights_only=True)
    for k in ("head4", "head2"):
        for name in a[k]:
            assert torch.equal(a[k][name], b[k][name]), f"{k}.{name} differs after resume"

    with pytest.raises(SystemExit, match="already finished"):
        trainer.train(_smoke_cfg(), tr, va, tmp_path / "res", prov)


def test_resume_refuses_different_data_or_config(tmp_path, monkeypatch):
    from scriptsense_ml.training import trainer

    real_init = MultiTaskDetector.__init__
    monkeypatch.setattr(MultiTaskDetector, "__init__",
                        lambda self, n, dropout=0.1, encoder_config=None: real_init(self, n, dropout, tiny_config()))
    tr, va = _frames()
    trainer.train(_smoke_cfg(stop_after_steps=2), tr, va, tmp_path / "r", {"train_row_ids_sha256": "abc"})
    with pytest.raises(SystemExit, match="different rows"):
        trainer.train(_smoke_cfg(), tr, va, tmp_path / "r", {"train_row_ids_sha256": "zzz"})
    with pytest.raises(SystemExit, match="learning_rate"):
        trainer.train(_smoke_cfg(learning_rate=5e-4), tr, va, tmp_path / "r", {"train_row_ids_sha256": "abc"})


def test_precision_never_picks_emulated_bf16(monkeypatch):
    from scriptsense_ml.training import trainer
    monkeypatch.setattr(trainer, "native_bf16_supported", lambda: False)  # e.g. Colab T4
    assert trainer.resolve_precision("auto", "cuda") == (torch.float16, True)
    monkeypatch.setattr(trainer, "native_bf16_supported", lambda: True)
    assert trainer.resolve_precision("auto", "cuda") == (torch.bfloat16, False)
    assert trainer.resolve_precision("auto", "cpu") == (None, False)


### `ml/tests/test_inference.py`

In [ ]:
%%writefile ml/tests/test_inference.py
import numpy as np
import pytest
from transformers import AutoTokenizer

from scriptsense_ml.inference.aggregation import Policy, aggregate
from scriptsense_ml.inference.chunking import eligible_spans, offset_to_page, segment_blocks, token_windows

PROSE = ("Researchers in the coastal town measured how quickly the tide eroded the old sea wall, "
         "and they compared those readings with records kept by the harbour office since the war. ")


@pytest.fixture(scope="module")
def tok():
    return AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")


def test_segment_blocks_excludes_headings_tables_and_references():
    text = ("Introduction\n\n" + PROSE * 2 + "\n\n| 1.2 | 3.4 | 5.6 | 7.8 | 9.0 | 1.1 | 2.2 | 3.3 |\n\n"
            + PROSE + "\n\nReferences\n\nSmith, J. (2020). A study of tides and walls in coastal towns. Journal.")
    blocks = segment_blocks(text)
    reasons = [b.reason for b in blocks]
    assert reasons == ["too_short", "", "non_prose", "", "references", "references"]
    spans = eligible_spans(blocks)
    assert len(spans) == 2  # windows never cross the excluded table
    for s, e in spans:
        assert text[s:e].startswith("Researchers")


def test_windows_cover_span_with_overlap_and_exact_offsets(tok):
    text = "Title\n\n" + PROSE * 40
    (span,) = eligible_spans(segment_blocks(text))
    wins = token_windows(text, span, tok, max_tokens=128, overlap=32)
    assert len(wins) > 3
    assert wins[0].start == span[0] and wins[-1].end == span[1]
    for a, b in zip(wins, wins[1:]):
        assert b.start < a.end  # overlap
        assert a.n_tokens <= 126
    # overlap-aware weights add up to exactly the covered characters
    assert sum(w.new_chars for w in wins) == span[1] - span[0]
    assert text[wins[1].start:wins[1].end] in text


def test_offset_to_page():
    assert [offset_to_page(o, [0, 100, 250]) for o in (0, 99, 100, 300)] == [1, 1, 2, 3]


def test_aggregate_insufficient_evidence():
    out = aggregate(np.zeros((0, 4)), np.zeros(0), np.zeros(0), 40, Policy(0.8, "fpr_0.01"))
    assert out["status"] == "insufficient_evidence" and out["class_probabilities"] is None


def test_aggregate_weighted_and_thresholded():
    p4 = np.array([[0.9, 0.05, 0.03, 0.02], [0.1, 0.1, 0.7, 0.1]])
    pai = np.array([0.1, 0.95])
    pol = Policy(threshold=0.8, operating_point="fpr_0.01")
    out = aggregate(p4, pai, np.array([3.0, 1.0]), 500, pol)
    assert out["status"] == "complete"
    assert out["flagged_text_share"] == pytest.approx(0.25)
    assert out["ai_signal_score"] == pytest.approx(0.75 * 0.1 + 0.25 * 0.95)
    assert out["document_class"] == "human_only"
    out = aggregate(p4, pai, np.array([1.0, 9.0]), 500, pol)
    assert out["document_class"] == "ai_paraphrased_or_refined"


def test_aggregate_disagreeing_heads_are_uncertain():
    out = aggregate(np.array([[0.2, 0.6, 0.1, 0.1]]), np.array([0.3]), np.array([1.0]), 500, Policy(0.8, "x"))
    assert out["document_class"] == "uncertain" and out["reasons"]


### `ml/tests/test_similarity.py`

In [ ]:
%%writefile ml/tests/test_similarity.py
from scriptsense_ml.similarity.fingerprints import align_exact, ngram_hashes, tokenize, winnow

SRC = ("The industrial revolution transformed rural economies as factories drew workers into rapidly "
       "growing cities, where housing shortages and poor sanitation created new public health crises.")


def test_tokenize_keeps_offsets():
    toks = tokenize("Hello, World!")
    assert [t.norm for t in toks] == ["hello", "world"]
    assert "Hello, World!"[toks[1].start:toks[1].end] == "World"


def test_winnow_guarantee_shared_run_shares_a_fingerprint():
    a = tokenize("intro words here " + SRC + " outro")
    b = tokenize("different start " + SRC)
    fa = {h for h, _ in winnow(ngram_hashes(a, 5), 4)}
    fb = {h for h, _ in winnow(ngram_hashes(b, 5), 4)}
    assert fa & fb


def test_align_exact_finds_copied_passage_with_offsets_case_insensitive():
    sub = "My essay begins here. " + SRC.upper() + " Then my own conclusion follows."
    m = align_exact(tokenize(sub), tokenize(SRC), n=5, min_words=8)
    assert len(m) == 1
    assert m[0].n_words == len(tokenize(SRC))
    assert sub[m[0].sub_start:m[0].sub_end].lower() == SRC.rstrip(".").lower()
    assert SRC[m[0].src_start:m[0].src_end] == SRC.rstrip(".")


def test_align_exact_ignores_short_common_phrases():
    sub = "on the other hand the results were mixed"
    src = "on the other hand the weather was fine"
    assert align_exact(tokenize(sub), tokenize(src), n=5, min_words=8) == []


def test_bm25_ranks_relevant_passage_first():
    from scriptsense_ml.similarity.lexical_search import BM25Index

    idx = BM25Index()
    idx.add("src:1", SRC)
    idx.add("src:2", "Glaciers retreat as global temperatures rise, exposing bedrock and meltwater lakes.")
    idx.add("src:3", "Recipes for sourdough bread require a starter culture and long fermentation.")
    hits = idx.search("factories drew rural workers into cities with poor sanitation", k=2)
    assert hits[0][0] == "src:1" and len(hits) == 1


## 3. Install dependencies

In [ ]:
if IN_COLAB:
    # Colab already ships CUDA torch, numpy and pandas; only the NLP stack is pinned.
    sh("-m", "pip", "install", "-q", "transformers==5.7.0", "tokenizers==0.22.2", "sentencepiece==0.2.2",
       "tiktoken", "pyarrow", "pyyaml", "scipy", "scikit-learn", "matplotlib")
else:
    # Locally, install CUDA torch first: pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cu128
    sh("-m", "pip", "install", "-q", "-r", "ml/requirements.txt")

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")

## 4. Unit tests
The tests use small synthetic fixtures only. They are never mixed into benchmark results.

In [ ]:
sh("-m", "pytest", "-q", "-p", "no:warnings", cwd="ml")

## 5. Prepared data (resume shortcut)
If a prepared dataset is already on Drive from an earlier session, it is restored here and sections 6–9 are skipped.

In [ ]:
# After a disconnect: restore the prepared dataset from Drive and skip sections 6-9.
DATA_READY = DATASET.exists()
if not DATA_READY and (BACKUP_DIR / "dataset.parquet").exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    print("Restoring prepared dataset from Drive ...")
    shutil.copy2(BACKUP_DIR / "dataset.parquet", DATASET)
    shutil.copytree(BACKUP_DIR / "manifests", "data/manifests", dirs_exist_ok=True)
    if (BACKUP_DIR / "dataset.parquet.leakage_ok.json").exists():  # cached leakage check for this exact file
        shutil.copy2(BACKUP_DIR / "dataset.parquet.leakage_ok.json", DATASET.parent)
    DATA_READY = True
print("Prepared dataset available:", DATA_READY, "-> sections 6-9 will be", "SKIPPED" if DATA_READY else "RUN")

## 6. Download datasets (hash-verified)
| Dataset | Size | License |
|---|---:|---|
| TXD-22 (Mendeley) | 121.7 MB | CC BY 4.0, attribution required |
| SemEval-2024 Task 8 (Zenodo) | 419.5 MB | **not stated in the record**: research use only, do not redistribute |
| RAID train.csv (Hugging Face) | **11.8 GB** | MIT per the dataset card |

In [ ]:
if not DATA_READY:
    sh("ml/scripts/download_datasets.py", "txd22", "semeval", "raid_train")

## 7. Profile raw datasets

In [ ]:
PROFILE_RAID = False   # profiling RAID is an extra full pass over 11.8 GB (~10 min); prepare_raid reports its counts anyway
if not DATA_READY:
    sh("ml/scripts/inspect_datasets.py", "txd22", "semeval", *(["raid"] if PROFILE_RAID else []))

## 8. Prepare canonical rows
RAID is streamed twice. Expect 20–40 minutes on Colab's 2 CPUs.

In [ ]:
if not DATA_READY:
    sh("ml/scripts/prepare_semeval.py")
    sh("ml/scripts/prepare_txd22.py")
    sh("ml/scripts/prepare_raid.py")

## 9. Quality filter, deduplication, grouped splits, leakage audit
This step fails hard if any lineage, prompt or duplicate group spans more than one split. Expect about 20–30 minutes on Colab. The result is backed up to Drive.

In [ ]:
if not DATA_READY:
    sh("ml/scripts/deduplicate_and_split.py")
    if SAVE_TO_DRIVE and IN_COLAB:
        print("Backing up prepared dataset to Drive (~0.9 GB) ...")
        BACKUP_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(DATASET, BACKUP_DIR / "dataset.parquet")
        shutil.copytree("data/manifests", BACKUP_DIR / "manifests", dirs_exist_ok=True)
    DATA_READY = True

# Run the leakage re-check once now (cached by dataset hash) and back up its result too.
if IN_COLAB and SAVE_TO_DRIVE:
    sh("-c", "import sys; sys.argv=['x']; sys.path.insert(0,'ml/scripts'); sys.path.insert(0,'ml'); "
             "import train_detector as t; from scriptsense_ml.training.data import load_dataset; "
             "from scriptsense_ml.data.io import file_sha256; p=t.Path(t.PROCESSED_DIR/'dataset.parquet'); "
             "t.check_text_leakage(load_dataset(p, ['text','split']), p, file_sha256(p))")
    leak = DATASET.parent / "dataset.parquet.leakage_ok.json"
    if leak.exists():
        BACKUP_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(leak, BACKUP_DIR)

In [ ]:
import pandas as pd
report_path = pathlib.Path("data/manifests/dataset_report.json")
if report_path.exists():
    rep = json.loads(report_path.read_text())
    print("rows before:", rep["rows_before"], "| after:", rep["rows_after"], "| train:", rep["train_rows_after_dedup"])
    print("dropped:", rep["dropped"])
    print("leaking keys:", rep["leakage"]["num_leaking_keys"])
    assert rep["leakage"]["num_leaking_keys"] == 0, "split leakage - do not train"
    display(pd.DataFrame(rep["distribution_after"]["class_4way_by_source_split"]).fillna(0).astype(int).T)
else:
    # The dataset was built in another workspace; train_detector.py still re-checks leakage before training.
    print("dataset_report.json is not in this workspace - summary skipped (training still re-checks leakage).")

## 10. GPU settings

In [ ]:
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if vram_gb >= 20:     # A100 / L4 / RTX 3090+
    GPU_SETS = ["per_device_batch_size=16", "grad_accum_steps=1", "freeze_word_embeddings=false"]
elif vram_gb >= 12:   # Colab T4 (15 GB): batch 8 x accumulation 2 = effective 16, comfortably within memory
    GPU_SETS = ["per_device_batch_size=8", "grad_accum_steps=2", "freeze_word_embeddings=false"]
else:                 # 4-6 GB laptop GPUs: measured on an RTX 3050 (3.5 GB peak)
    GPU_SETS = ["per_device_batch_size=4", "grad_accum_steps=4", "freeze_word_embeddings=true"]
print(f"GPU: {torch.cuda.get_device_name(0) if vram_gb else 'none'} ({vram_gb:.1f} GB) ->", GPU_SETS)

## 11. Smoke training run (documented subset)
This uses 3,000 train rows, 1,000 validation rows and 1 epoch. It **only checks that the pipeline works on this GPU**. Its metrics are not model results.

In [ ]:
SMOKE_RUN = "smoke_deberta_mt_s1"
if not (ART_DIR / "runs" / SMOKE_RUN / "best" / "calibration.json").exists():
    if not run_finished(SMOKE_RUN):
        sh("ml/scripts/train_detector.py", "--set", f"run_name={SMOKE_RUN}", "max_train_rows=3000",
           "max_val_rows=1000", "epochs=1", "checkpoint_every_steps=0", *GPU_SETS)
    sh("ml/scripts/calibrate_thresholds.py", "--run", SMOKE_RUN)
    sh("ml/scripts/evaluate_detector.py", "--run", SMOKE_RUN, "--max-rows", "1000")
else:
    print("Smoke run already done - skipped.")

## 12. Full training run
This trains on all training rows for up to 3 epochs, with early stopping on validation. It then calibrates on validation only and evaluates on the held-out sets.
* A checkpoint is saved to Drive every 1,000 steps and at each epoch end. **After a disconnect, just Run all again** and it resumes.
* The log shows `step` / `of_steps`, so you can follow progress.
* For the remaining protocol runs (RoBERTa baseline, other seeds, ablations), change `RUN_NAME` and add `--set` values, e.g.
  `model_name=roberta-base mode=binary`, `seed=2`, or `ablation={exclude_raid_paraphrase: true}`.

In [ ]:
RUN_FULL = True                      # set False to stop after the smoke test
RUN_NAME = "deberta_multitask_s1"    # keep the same name so a re-run RESUMES this run

if RUN_FULL:
    if not run_finished(RUN_NAME):
        sh("ml/scripts/train_detector.py", "--set", f"run_name={RUN_NAME}", *GPU_SETS)   # resumes automatically
    if not (ART_DIR / "runs" / RUN_NAME / "best" / "calibration.json").exists():
        sh("ml/scripts/calibrate_thresholds.py", "--run", RUN_NAME)
    if not (ART_DIR / "runs" / RUN_NAME / "eval" / "metrics_summary.csv").exists():
        sh("ml/scripts/evaluate_detector.py", "--run", RUN_NAME)
    print("Full run complete:", ART_DIR / "runs" / RUN_NAME)

## 13. Results
Fill the model card only from these files.

In [ ]:
import pandas as pd
from IPython.display import Image, display
RUN_NAME = globals().get("RUN_NAME", "deberta_multitask_s1")
run = RUN_NAME if (ART_DIR / "runs" / RUN_NAME / "eval").exists() else "smoke_deberta_mt_s1"
eval_dir = ART_DIR / "runs" / run / "eval"
print("run:", run, "(smoke run = pipeline check only, not a model result)" if run.startswith("smoke") else "")
display(pd.read_csv(eval_dir / "metrics_summary.csv"))
for png in sorted(eval_dir.glob("*.png")):
    print(png.name); display(Image(filename=str(png)))

## 14. Document-level inference demo
This produces the same response shape as the FastAPI `/api/v2/documents/{id}/ai-analysis` endpoint.

In [ ]:
sys.path.insert(0, str(WORK_DIR / "ml"))
from scriptsense_ml.inference.service import AIWritingAnalyzer
analyzer = AIWritingAnalyzer(ART_DIR / "runs" / run / "best")
sample = ("Paste or load any English document here. It needs at least 150 words of eligible prose; "
          "shorter input returns insufficient_evidence instead of a score. ") * 15
result = analyzer.analyze(sample, page_starts=[0], document_id="notebook-demo")
{k: result[k] for k in ["status", "document_class", "class_probabilities", "ai_signal_score",
                        "flagged_text_share", "eligible_word_count", "model_version", "limitations"]}